# Synthetic data using CEHR-GPT — COPD cohort (FULL)

End-to-end on the **full** cohort: stage OMOP tables from Foundry → build training sequences → train CEHR-GPT (resumable across 6-hour GPU sessions) → generate synthetic patients → convert back to OMOP.

**After any overnight wipe:** run the Environment cells (Cell 1 → reset pytorch → torch CUDA layer → patch), then restore data/checkpoints from Foundry, then continue from Step 1 / Step 2.


## Environment — run after every wipe
Cell 1 rebuilds the managed env via Maestro; then reset pytorch to a plain spec; then layer the **CUDA** build of torch as a constraints-pinned pip layer; then re-apply the local `gpt_utils` patch. Restart the kernel after Cell 1 if it installs anything.


In [ ]:
"""
CELL 1 - ENVIRONMENT DIAGNOSTIC + AUTO-INSTALL via Maestro's own CLI.
Does NOT edit meta.yaml or lock files. For anything missing or mis-versioned it runs
the same command the GUI runs:   maestro env conda install "<spec>"
Set AUTO_INSTALL = False to make this cell report-only.
After an install round: RESTART THE KERNEL, then re-run this cell until all-OK.

Reference target env (also what this cell enforces):
  conda: openjdk==17.0.18 | pytorch>=2.4 | numpy>=1.24,<2 | pandas>=2.0,<2.2 (NOT 2.2.0)
  rest:  pyspark==3.5.5 | pyarrow | pyarrow-hotfix | datasets==2.16.1 | transformers==4.44.1
         tokenizers==0.19.0 | accelerate>=0.31.0 | peft==0.10.0 | optuna==4.0.0
         evaluate==0.4.1 | scikit-learn==1.4.0 | scipy==1.12.0 | xgboost==2.0.3
         dask==2024.1.1 | meds==0.3.3 | meds_reader==0.1.9 | femr>=0.2.0,<=0.2.4
         fast-ml==3.68 | openai==1.54.3 | lightgbm | polars | wandb>=0.17.8
         jinja2==3.1.3 | networkx>=3.2.1 | pillow>=10.3.0 | python-dateutil==2.8.2
         pyyaml==6.0.1 | tqdm>=4.66.1 | werkzeug==3.0.1 | setuptools<80
NEVER install: cehrbert, cehrbert_data, cehrgpt (local folders; a pip copy shadows them).
NEVER put meds/meds_reader/femr/fast-ml in the GUI pip section - their declared deps
conflict with the pins (meds_reader wants pandas>=2.2); this cell layers them --no-deps.
Python itself: pins were validated on 3.11 - this cell will NOT change python;
if the solver fights you on 3.12, pin python==3.11 in the GUI.
"""

import os, sys, subprocess, importlib, importlib.util, importlib.machinery
import importlib.metadata as md
from pathlib import Path

AUTO_INSTALL = True          # set False to diagnose only

OK, BAD = [], []
def check(name, good, detail=""):
    (OK if good else BAD).append((name, detail))
    print(f"  {'OK  ' if good else 'FIX '} {name:30s} {detail}")

# ---------- runtime env vars (process-level only) ----------
JVM_OPTS = " ".join(f"--add-opens=java.base/{m}=ALL-UNNAMED" for m in [
    "java.lang","java.lang.invoke","java.lang.reflect","java.io","java.net","java.nio",
    "java.util","java.util.concurrent","sun.nio.ch","sun.nio.cs","sun.security.action","sun.util.calendar"])
os.environ["JAVA_TOOL_OPTIONS"] = os.environ["_JAVA_OPTIONS"] = JVM_OPTS
os.environ["JAVA_HOME"] = sys.prefix
if f"{sys.prefix}/bin" not in os.environ["PATH"]:
    os.environ["PATH"] = f"{sys.prefix}/bin:" + os.environ["PATH"]

print(f"  python {sys.version.split()[0]}  (pins originally validated on 3.11; 3.12 wheels confirmed working)")

# ---------- package checks ----------
try:
    from packaging.specifiers import SpecifierSet
    from packaging.version import Version
    def meets(v, spec): return Version(v) in SpecifierSet(spec)
except ImportError:
    def meets(v, spec): return True

# (dist name, version spec, conda name if it differs)
PINS = [
    ("pyspark","==3.5.5",None), ("numpy",">=1.24,<2",None), ("pandas",">=2.0,<2.2",None),
    ("pyarrow",None,None), ("torch",">=2.4.0","pytorch"), ("transformers","==4.44.1",None),
    ("tokenizers","==0.19.0",None), ("datasets","==2.16.1",None), ("accelerate",">=0.31.0",None),
    ("peft","==0.10.0",None), ("optuna","==4.0.0",None), ("evaluate","==0.4.1",None),
    ("scikit-learn","==1.4.0",None), ("scipy","==1.12.0",None), ("xgboost","==2.0.3",None),
    ("dask","==2024.1.1",None), ("meds","==0.3.3",None), ("meds_reader","==0.1.9",None),
    ("femr",">=0.2.0,<=0.2.4",None), ("openai","==1.54.3",None), ("lightgbm",None,None),
    ("polars",None,None), ("wandb",">=0.17.8",None), ("packaging",">=23.2",None),
    ("pyarrow_hotfix",None,"pyarrow-hotfix"), ("foundry-transforms-lib-python",None,None),
    ("fast-ml","==3.68",None), ("Jinja2","==3.1.3","jinja2"), ("networkx",">=3.2.1",None),
    ("Pillow",">=10.3.0","pillow"), ("python-dateutil","==2.8.2",None), ("PyYAML","==6.0.1","pyyaml"),
    ("tqdm",">=4.66.1",None), ("Werkzeug","==3.0.1","werkzeug"), ("setuptools","<80.0.0",None),
    ("dill","<0.3.8",None), ("multiprocess","==0.70.15",None),
    ("fsspec","<=2023.10.0",None), ("huggingface-hub","<1.0","huggingface_hub"),
]
PIP_ONLY_LIKELY = {"meds","meds_reader","femr","fast-ml"}   # probably absent from conda channels

to_install = []   # (spec string for maestro, is_pip_only_likely)
print("=== packages ===")
for dist, spec, conda_name in PINS:
    cname = conda_name or dist
    try:
        v = md.version(dist)
        if v is None:                      # mangled dist-info (aborted install)
            raise md.PackageNotFoundError(dist)
        good = spec is None or meets(v, spec)
        check(dist, good, f"{v}" + (f"  -> {cname}{spec}" if not good else ""))
        if not good: to_install.append((cname + (spec or ""), dist in PIP_ONLY_LIKELY))
    except Exception:
        check(dist, False, f"missing/broken -> {cname}{spec or ''}")
        to_install.append((cname + (spec or ""), dist in PIP_ONLY_LIKELY))

TRANSITIVE = ["regex","joblib","threadpoolctl","filelock","fsspec","dill","multiprocess",
              "xxhash","huggingface_hub","safetensors","aiohttp","psutil","requests"]
miss = [t for t in TRANSITIVE if importlib.util.find_spec(t) is None]
check("transitive deps", not miss,
      f"missing: {', '.join(miss)} (should arrive as deps of the batch)" if miss else "all present")

# ---------- java / spark ----------
print(); print("=== java / spark ===")
def run_quiet(cmd):
    try: return subprocess.run(cmd, capture_output=True, text=True)
    except FileNotFoundError: return None
r = run_quiet(["java","-version"])
_out = ((r.stderr or "") + (r.stdout or "")) if r else ""
_lines = [l for l in _out.splitlines() if l.strip() and not l.startswith("Picked up")]
line = next((l for l in _lines if "version" in l.lower()), _lines[0] if _lines else "java binary not found")
good_java = bool(r) and '"17' in line
check("openjdk 17", good_java, line)
if not good_java: to_install.append(("openjdk==17.0.18", False))
r = run_quiet(["which","spark-submit"])
check("spark-submit on PATH", bool(r and r.stdout.strip()),
      (r.stdout.strip() if r and r.stdout.strip() else "missing (arrives with pyspark==3.5.5)"))
try:
    import pyspark
    cores = list((Path(pyspark.__file__).parent/"jars").glob("spark-core_*.jar"))
    check("pyspark jars coherent", len(cores)==1 and "3.5.5" in cores[0].name,
          cores[0].name if cores else "no spark-core jar")
except ModuleNotFoundError:
    check("pyspark jars coherent", False, "pyspark not installed yet")
except Exception as e:
    check("pyspark jars coherent", False, repr(e))

# ---------- local cehr packages (folders, never pip) ----------
print(); print("=== local cehr packages ===")
LOCAL = {"cehrbert_data": "/home/user/repo/cehrbert_data-main/src",
         "cehrgpt":       "/home/user/repo/cehrgpt-main/src",
         "cehrbert":      "/home/user/repo/cehrbert-main"}
for mod, root in LOCAL.items():
    spec = importlib.machinery.PathFinder().find_spec(mod, [root])
    check(f"local {mod}", bool(spec and spec.origin), (spec.origin if spec else f"missing under {root}"))
shadows = []
for d in ("cehrbert-data","cehrbert","cehrgpt"):
    try: md.distribution(d); shadows.append(d)
    except md.PackageNotFoundError: pass
check("no pip-installed cehr* shadows", not shadows,
      f"{shadows} installed -> REMOVE via GUI" if shadows else "")
rogue = Path("/home/user/repo/cehrbert_data")
check("no rogue /home/user/repo/cehrbert_data", not rogue.exists(),
      "exists - move it aside, it can shadow imports" if rogue.exists() else "")

# ---------- data dirs (idempotent) ----------
for d in ["omop_synthea","cehrgpt_output_full","model_run_full"]:
    os.makedirs(f"/home/user/repo/{d}", exist_ok=True)

# ---------- AUTO-INSTALL via Maestro's own CLI (same command the GUI runs) ----------
def maestro_cmd(args, label):
    cmd = ["maestro", "env"] + args
    print(f"\n>>> {label}:\n    {' '.join(cmd)}", flush=True)
    try:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             stdin=subprocess.DEVNULL, text=True, bufsize=1)
    except FileNotFoundError:
        print("    'maestro' CLI not found on PATH"); return False
    for ln in p.stdout: print(ln, end="")
    rc = p.wait(); print(f"    rc={rc}")
    return rc == 0

def maestro_install(specs, label, mode="conda"):
    if not specs: return True
    return maestro_cmd([mode, "install"] + specs, label)

# The 4 niche pkgs must NEVER live in the managed pip section: meds_reader declares
# pandas>=2.2 and femr declares open-ended transformers/datasets/sklearn/scipy, so every
# `maestro env install` pip phase force-reinstalls them WITH deps and clobbers our pins.
# The original working env installed them `pip --no-deps` - we reproduce exactly that:
# a no-deps layer applied AFTER each sync, never written into the spec.
NICHE_NO_DEPS = ["meds==0.3.3", "meds_reader==0.1.9", "femr==0.2.4", "fast-ml==3.68"]
REPAIR_EXACT  = {"pandas": "pandas==2.1.4", "transformers": "transformers==4.44.1",
                 "tokenizers": "tokenizers==0.19.0", "datasets": "datasets==2.16.1",
                 "scikit-learn": "scikit-learn==1.4.0", "scipy": "scipy==1.12.0",
                 "Jinja2": "jinja2==3.1.3", "python-dateutil": "python-dateutil==2.8.2",
                 "PyYAML": "pyyaml==6.0.1", "numpy": "numpy==1.26.4",
                 "dill": "dill==0.3.7", "multiprocess": "multiprocess==0.70.15",
                 "fsspec": "fsspec==2023.10.0", "huggingface-hub": "huggingface-hub==0.24.7"}

def pip_no_deps(specs, force=False):
    if not specs: return True
    cmd = ([sys.executable, "-m", "pip", "install", "--no-deps"]
           + (["--force-reinstall"] if force else []) + specs)
    print(f"\n>>> pip --no-deps{' --force-reinstall' if force else ''}: {' '.join(specs)}", flush=True)
    p = subprocess.run(cmd, capture_output=True, text=True)
    print((p.stdout + p.stderr).strip()[-1200:])
    print(f"    rc={p.returncode}")
    return p.returncode == 0

def broken_pins():
    importlib.invalidate_caches()
    out = []
    for dist, spec, conda_name in PINS:
        if dist in REPAIR_EXACT and spec:
            try:
                v = md.version(dist)
                if v is None or not meets(v, spec): out.append(REPAIR_EXACT[dist])
            except Exception:
                out.append(REPAIR_EXACT[dist])
    return out

print(); print("="*64)
if to_install and AUTO_INSTALL:
    # 0) pull the niche pkgs OUT of the managed pip section (harmless no-op if absent)
    maestro_cmd(["pip", "uninstall", "meds", "meds_reader", "femr", "fast-ml"],
                "removing niche pkgs from managed pip section (their deps clobber the pins)")
    # 1) conda batch for everything else
    main_specs = [s for s, piponly in to_install if not piponly]
    ok_main = maestro_install(main_specs, f"installing main batch ({len(main_specs)} specs, one solve)")
    # 1b) pre-sync heal: fix stale pip-layer pins (femr-era residue like dill==0.4.1 that
    #     contradicts datasets 2.16.1), then make a benign spec op so Hawk re-snapshots the
    #     lockfile from the now-healthy layer - otherwise the sync replay is unsolvable
    pre = broken_pins()
    if pre:
        print(f"\npre-sync repair of stale pip-layer pins: {pre}")
        pip_no_deps(pre, force=True)
        maestro_cmd(["pip", "install", "setuptools<80.0.0"], "re-snapshot pip lockfile (benign spec op)")
    # 2) full sync: Hawk materializes the env from the (now niche-free) spec
    ok_sync = maestro_cmd(["install"], "final environment sync")
    # 3) niche pkgs layered on top with --no-deps (mirrors the original working env)
    pip_no_deps(NICHE_NO_DEPS)
    # 4) repair pass: force back any pinned package the sync's pip phase clobbered
    broken = broken_pins()
    if broken:
        print(f"\nrepairing clobbered pins: {broken}")
        pip_no_deps(broken, force=True)
    print(); print("="*64)
    if not ok_sync:
        print("Final 'maestro env install' sync FAILED - read the output above before continuing.")
    if not ok_main:
        print("Main batch FAILED - read the solver output above.")
    print("\nNow RESTART THE KERNEL and re-run this cell until everything is OK.")
elif to_install:
    print(f"AUTO_INSTALL=False - {len(to_install)} item(s) to fix:")
    for s, _ in to_install: print(f"  - {s}")
elif BAD:
    print("No installs needed, but some checks failed - see FIX lines above.")
else:
    print("Environment matches target - proceed to the pipeline cells.")


In [ ]:
import subprocess
def maestro(*a):
    p=subprocess.Popen(["maestro","env",*a],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,stdin=subprocess.DEVNULL,text=True,bufsize=1)
    [print(l,end="") for l in p.stdout]; print("  rc=",p.wait())
maestro("conda","install","pytorch>=2.4")   # reset pytorch back to a plain spec

In [ ]:
import sys, subprocess
open("/tmp/pins.txt","w").write("""numpy==1.26.4
                                    jinja2==3.1.3
                                    fsspec==2023.10.0
                                    pandas==2.1.4
                                    transformers==4.44.1
                                    tokenizers==0.19.0
                                    datasets==2.16.1
                                    scikit-learn==1.4.0
                                    scipy==1.12.0
                                    huggingface-hub==0.24.7
                                    dill==0.3.7
                                    multiprocess==0.70.15
                                    """)

r = subprocess.run([sys.executable,"-m","pip","install","torch==2.12.0","-c","/tmp/pins.txt"], capture_output=True, text=True)
print(r.stdout[-2500:]);
if r.returncode: print("ERR:\n", r.stderr[-2000:])

In [ ]:
import os, sys, subprocess

print("="*64, "\n1) nvidia-smi (driver + GPU present?)\n" + "="*64)
subprocess.run(["bash","-c","nvidia-smi || echo 'NO GPU / nvidia-smi missing'"])

print("\n" + "="*64, "\n2) torch in THIS kernel\n" + "="*64)
try:
    import torch
    print("torch", torch.__version__, "| built for CUDA", torch.version.cuda)
    ok = torch.cuda.is_available(); print("cuda available:", ok)
    if ok:
        print("count:", torch.cuda.device_count(), "| name:", torch.cuda.get_device_name(0))
        print("capability:", torch.cuda.get_device_capability(0), "| bf16:", torch.cuda.is_bf16_supported())
        free, tot = torch.cuda.mem_get_info(); print(f"GPU mem: {free/1e9:.1f} free / {tot/1e9:.1f} GB")
        import time; x = torch.randn(8192, 8192, device="cuda"); torch.cuda.synchronize()
        t=time.time()
        for _ in range(10): y = x @ x
        torch.cuda.synchronize(); print(f"matmul on GPU: OK ({time.time()-t:.3f}s for 10x 8192^3)")
    else:
        print(">>> kernel torch does NOT see the GPU")
except Exception as e:
    print("torch failed:", repr(e))

print("\n" + "="*64, "\n3) does the TRAINING subprocess env see the GPU?\n" + "="*64)
PKG = ["/home/user/repo/cehrbert-main","/home/user/repo/cehrgpt-main/src","/home/user/repo/cehrbert_data-main/src"]
probe = "import torch; a=torch.cuda.is_available(); print('cuda:', a, '| dev:', torch.cuda.get_device_name(0) if a else None)"
for label, ld in [("WITH conda-lib prepend (what training uses)", f"{sys.prefix}/lib:" + os.environ.get("LD_LIBRARY_PATH","")),
                  ("WITHOUT conda-lib prepend",                   os.environ.get("LD_LIBRARY_PATH",""))]:
    e = os.environ.copy(); e.pop("CUDA_VISIBLE_DEVICES", None)
    e["LD_LIBRARY_PATH"] = ld; e["PYTHONPATH"] = ":".join(PKG + [e.get("PYTHONPATH","")])
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True, env=e)
    print(f"  {label}:\n    {r.stdout.strip() or r.stderr.strip()[-300:]}")

In [ ]:
import subprocess, sys, os
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = f"{sys.prefix}/lib:" + env.get("LD_LIBRARY_PATH", "")
env["PYTHONPATH"] = ":".join([
    "/home/user/repo/cehrbert-main",
    "/home/user/repo/cehrgpt-main/src",
    "/home/user/repo/cehrbert_data-main/src",
    env.get("PYTHONPATH", ""),
])
r = subprocess.run([sys.executable, "-c", """
import cehrbert_data; print('cehrbert_data', cehrbert_data.__file__)
import cehrbert; print('cehrbert', cehrbert.__file__)
import cehrgpt; print('cehrgpt', cehrgpt.__file__)
from cehrbert.data_generators.hf_data_generator import meds_utils
import cehrgpt.runners.hf_cehrgpt_pretrain_runner
print('full import chain OK')
"""], capture_output=True, text=True, env=env)
print("STDOUT:")
print(r.stdout)
print("STDERR:")
print(r.stderr[-5000:])

In [ ]:
# robustness patch: let the tokenizer survive malformed/negative time tokens like "i-D-6"
patch_path = "/home/user/repo/cehrgpt-main/src/cehrgpt/gpt_utils.py"
MARK = "robustness patch: tolerate malformed time tokens"
override = f'''

# {MARK}  (overrides extract_time_interval_in_days for tokens like "i-D-6")
import re as _re_patch
def extract_time_interval_in_days(token):  # noqa: F811
    try:
        return int(token.split("-")[1][1:])            # original fast path: "i-D6" -> 6
    except (IndexError, ValueError):
        m = _re_patch.search(r"-?\\d+$", str(token))     # trailing signed int: "i-D-6" -> -6
        return max(0, int(m.group())) if m else 0        # clamp negatives/unparseable to 0
'''
src = open(patch_path).read()
if MARK not in src:
    with open(patch_path, "a") as f: f.write(override)
    print("patched:", patch_path)
else:
    print("already patched")

## Step 0 — Stage OMOP data from Foundry (full cohort)
Downloads every table into `omop_synthea/` under canonical OMOP names (file-level, memory-safe). Idempotent: complete tables are skipped.


In [ ]:
from foundry.transforms import Dataset
from pathlib import Path
import shutil, time, traceback

DATASETS = {
    "copd_concept_parquet":              ("concept",              4),
    "copd_ancestor_concept_copd":        ("concept_ancestor",     5),
    "copd_relationship_concept_parquet": ("concept_relationship", 3),
    "copd_condition_occurrence_parquet": ("condition_occurrence", 200),
    "copd_drug_exposure_parquet":        ("drug_exposure",        526),
    "copd_measurement_parquet":          ("measurement",          2674),
    "copd_observation_parquet":          ("observation",          236),
    "copd_procedure_occurrence_parquet": ("procedure_occurrence", 71),
    "copd_visit_occurrence_parquet":     ("visit_occurrence",     67),
    "copd_death_parquet":                ("death",                4), 
    "person":                            ("person",               4),
}

base_dir = Path("/home/user/repo/omop_synthea")
def log(m): print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)

for fd_name, (omop_name, expected) in DATASETS.items():
    out_dir = base_dir / omop_name
    have = len(list(out_dir.glob("*.parquet"))) if out_dir.is_dir() else 0
    if have >= expected:
        log(f"SKIP  {omop_name:24s} ({have} parquet files present)"); continue
    log(f"STAGE {omop_name:24s} have {have}, need {expected} -- downloading {fd_name}")
    if out_dir.is_file(): out_dir.unlink()
    elif out_dir.exists(): shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)
    try:
        downloaded = Dataset.get(fd_name).files().download()
        n = size = 0
        for src_path in downloaded.values():
            src = Path(src_path)
            if src.suffix == ".parquet":
                shutil.copy(src, out_dir / src.name); n += 1; size += src.stat().st_size
        (out_dir / "_SUCCESS").touch()
        log(f"      done: {n} parquet files, {size/1e9:.2f} GB")
    except Exception as e:
        log(f"      FAILED: {type(e).__name__}: {e}"); traceback.print_exc()

In [ ]:
import os
BASE = "/home/user/repo/omop_synthea"
EXPECT = {"concept":4,"concept_ancestor":5,"concept_relationship":3,"condition_occurrence":200,
          "drug_exposure":526,"measurement":2674,"observation":236,"procedure_occurrence":71,
          "visit_occurrence":67, "death":4,"person":4}
ok = True
for t, want in EXPECT.items():
    d = os.path.join(BASE, t)
    n = sum(f.endswith(".parquet") for _,_,fs in os.walk(d) for f in fs) if os.path.isdir(d) else 0
    print(f"  {'OK  ' if n >= want else 'FAIL'}  {t:24s} {n:5d} / {want}")
    ok = ok and n >= want
print("\nREADY" if ok else "\nNOT READY -- re-run the download cell")


In [ ]:
import glob, shutil, os, pyarrow.parquet as pq, pyarrow.compute as pc
OMOP="/home/user/repo/omop_synthea"

# 1) measurement must carry real visit links, or the handler drops every measurement row
mf=sorted(glob.glob(f"{OMOP}/measurement/*.parquet"))
mcols=set(pq.read_schema(mf[0]).names)
print("visit_occurrence_id column:", "visit_occurrence_id" in mcols)
if "visit_occurrence_id" in mcols:
    tot=nn=0
    for f in mf[::100]:
        v=pq.read_table(f, columns=["visit_occurrence_id"]).column(0)
        tot+=len(v); nn+=len(v)-(pc.sum(pc.is_null(v)).as_py() or 0)
    print(f"  non-null: {100*nn/max(1,tot):.1f}%  (need >0)")

# 2) clear stale processed_* caches so the qcl reprocesses cleanly
for d in glob.glob(f"{OMOP}/processed_*"):
    if not glob.glob(f"{d}/*.parquet"): shutil.rmtree(d); print("cleared stale cache:", os.path.basename(d))
print("preflight done")

In [ ]:
# Quiet Spark runner: full output -> logfile, notebook shows only errors + heartbeat.
# Also writes a Log4j2 config (Spark 3.5 IGNORES the old log4j.properties - it is log4j 1.x format).
import subprocess, sys, os, time, re
from pathlib import Path

LOGDIR = Path("/home/user/repo/logs"); LOGDIR.mkdir(exist_ok=True)

LOG4J2 = "/home/user/repo/log4j2.properties"
Path(LOG4J2).write_text(
    "rootLogger.level = warn\n"
    "rootLogger.appenderRef.console.ref = console\n"
    "appender.console.type = Console\n"
    "appender.console.name = console\n"
    "appender.console.target = SYSTEM_ERR\n"
    "appender.console.layout.type = PatternLayout\n"
    "appender.console.layout.pattern = %d{yy/MM/dd HH:mm:ss} %p %c{1}: %m%n\n"
)
LOG4J2_OPT = f"-Dlog4j2.configurationFile=file:{LOG4J2}"   # append to --driver-java-options

IMPORTANT = re.compile(r"ERROR|Exception|Traceback|FAILED|py4j\.protocol|File \"|rc=", re.I)

def run_spark(cmd, env=None, name="spark_job", heartbeat_every=500):
    """Run a spark-submit command quietly. Returns the exit code."""
    log_path = LOGDIR / f"{name}-{time.strftime('%Y%m%d-%H%M%S')}.log"
    print(f"logfile: {log_path}")
    print("cmd    :", " ".join(map(str, cmd))[:240], "...")
    n = shown = 0; t0 = time.time()
    with open(log_path, "w") as lf:
        p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env=env)
        for line in p.stdout:
            lf.write(line); n += 1
            if IMPORTANT.search(line):
                print(line, end=""); shown += 1
            elif n % heartbeat_every == 0:
                print(f"  ... {n} log lines, {time.time()-t0:.0f}s elapsed", flush=True)
    rc = p.wait()
    print(f"\nrc={rc}  ({n} lines total, {shown} shown, full log -> {log_path})")
    if rc != 0:
        print("--- last 30 log lines ---")
        print("".join(open(log_path).readlines()[-30:]))
    return rc


## Step 1 — Build training sequences (full cohort)
Qualified concept list → patient sequences → cache to Foundry. Domains: condition / drug / procedure (measurement is excluded — the staged export lacks the columns cehrgpt's measurement handler requires).


In [ ]:
import os
os.environ['OMOP_DIR']           = '/home/user/repo/omop_synthea'
os.environ['CEHR_GPT_DATA_DIR']  = '/home/user/repo/cehrgpt_output_full'
os.environ['CEHR_GPT_MODEL_DIR'] = '/home/user/repo/model_run_full'
os.makedirs(os.environ['CEHR_GPT_DATA_DIR'], exist_ok=True)
os.makedirs(os.environ['CEHR_GPT_MODEL_DIR'], exist_ok=True)
print({k: os.environ[k] for k in ('OMOP_DIR','CEHR_GPT_DATA_DIR','CEHR_GPT_MODEL_DIR')})


In [ ]:
import glob, pyarrow as pa, pyarrow.parquet as pq
mfiles = sorted(glob.glob("/home/user/repo/omop_synthea/measurement/*.parquet"))
ADD = {"measurement_datetime": pa.timestamp("us"), "unit_source_value": pa.string()}  # NOT visit_occurrence_id
done = 0
for f in mfiles:
    t = pq.read_table(f)
    if set(ADD) <= set(t.column_names): done += 1; continue
    cols = {n: t[n] for n in t.column_names}          # keeps real visit_occurrence_id
    for name, typ in ADD.items():
        if name not in cols: cols[name] = pa.nulls(t.num_rows, typ)
    pq.write_table(pa.table(cols), f)
    done += 1
    if done % 300 == 0: print(f"  ...{done}/{len(mfiles)}")
print(f"patched {done} files -> columns now:", pq.read_schema(mfiles[0]).names)

In [ ]:
import shutil; shutil.rmtree("/home/user/repo/omop_synthea/processed_measurement", ignore_errors=True); print("cleared cache")

In [ ]:
import subprocess, os, sys, textwrap, shutil, time, re, glob
from pathlib import Path
import pyarrow.parquet as pq

OMOP = os.environ["OMOP_DIR"]
assert glob.glob(f"{OMOP}/concept/*.parquet") and glob.glob(f"{OMOP}/condition_occurrence/*.parquet")

# --- preflight: measurement must be staged AND carry the columns the handler needs ---
mfiles = glob.glob(f"{OMOP}/measurement/*.parquet")
assert mfiles, "measurement not staged — run the Step 0 staging cell first"
mcols = set(pq.read_schema(mfiles[0]).names)
date_ok = bool({"measurement_date", "measurement_datetime"} & mcols)
val_ok  = "value_as_number" in mcols
print("measurement columns:", sorted(mcols))
print(f"  measurement_concept_id: {'OK' if 'measurement_concept_id' in mcols else 'MISSING'}"
      f" | date: {'OK' if date_ok else 'MISSING'} | value_as_number: {'OK' if val_ok else 'MISSING'}")
if not ("measurement_concept_id" in mcols and date_ok):
    raise SystemExit("measurement export lacks concept_id/date — fix the export before adding it (this is why it was excluded).")
if not val_ok:
    print("  ⚠ no value_as_number — measurements will lack lab values but concepts will still be included")

shutil.rmtree(f"{OMOP}/qualified_concept_list", ignore_errors=True)

LAUNCHER = "/home/user/repo/_cehr_launcher_cl.py"
Path(LAUNCHER).write_text(textwrap.dedent(r'''
    import sys, importlib, runpy
    REAL = ["/home/user/repo/cehrbert_data-main/src","/home/user/repo/cehrbert-main","/home/user/repo/cehrgpt-main/src"]
    for p in reversed(REAL):
        while p in sys.path: sys.path.remove(p)
        sys.path.insert(0, p)
    for m in [m for m in list(sys.modules) if m=="cehrbert_data" or m.startswith("cehrbert_data.")]:
        del sys.modules[m]
    importlib.invalidate_caches()
    APP = "/home/user/repo/cehrbert_data-main/src/cehrbert_data/apps/generate_included_concept_list.py"
    sys.argv = [APP] + sys.argv[1:]
    runpy.run_path(APP, run_name="__main__")
'''))

JVM_OPTS = ("--add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
            "--add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED "
            "--add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED")
LOG4J2_OPT = "-Dlog4j2.configurationFile=file:/home/user/repo/log4j2.properties"
PKG = "/home/user/repo/cehrbert_data-main/src:/home/user/repo/cehrbert-main:/home/user/repo/cehrgpt-main/src"
env = os.environ.copy()
env["JAVA_TOOL_OPTIONS"] = env["_JAVA_OPTIONS"] = JVM_OPTS
env["JAVA_HOME"] = sys.prefix
env["PATH"] = f"{sys.prefix}/bin:" + env.get("PATH", "")
env["LD_LIBRARY_PATH"] = f"{sys.prefix}/lib:" + env.get("LD_LIBRARY_PATH", "")
env["PYTHONPATH"] = PKG + ":" + env.get("PYTHONPATH", "")

os.makedirs("/home/user/repo/spark_tmp", exist_ok=True)
cmd = ["spark-submit", "--master", "local[16]", "--driver-memory", "48g",
       "--conf", "spark.sql.shuffle.partitions=200", "--conf", "spark.sql.adaptive.enabled=true",
       "--conf", "spark.local.dir=/home/user/repo/spark_tmp",
       "--conf", "spark.serializer=org.apache.spark.serializer.KryoSerializer",
       "--driver-java-options", f"{JVM_OPTS} {LOG4J2_OPT}",
       LAUNCHER, "-i", OMOP, "-o", OMOP,
       "--min_num_of_patients", "100",
       "--ehr_table_list", "condition_occurrence", "drug_exposure", "procedure_occurrence", "measurement"]

LOGDIR = Path("/home/user/repo/logs"); LOGDIR.mkdir(exist_ok=True)
log_path = LOGDIR / f"qcl_full-{time.strftime('%Y%m%d-%H%M%S')}.log"
KEEP = re.compile(r'ERROR|Exception|Traceback|FAILED|py4j\.protocol|File "', re.I)
print(f"logfile: {log_path}\n")
n = 0; t0 = time.time()
with open(log_path, "w") as lf:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        lf.write(line); n += 1
        if KEEP.search(line): print(line, end="")
        elif n % 500 == 0: print(f"  ... {n} lines, {(time.time()-t0)/60:.1f} min", flush=True)
rc = p.wait()
print(f"\nrc={rc}  ({n} lines, {(time.time()-t0)/60:.1f} min)")
if rc != 0:
    print("".join(open(log_path).readlines()[-30:]))
else:
    fs = glob.glob(f"{OMOP}/qualified_concept_list/*.parquet")
    print(f"qualified_concept_list -> {len(fs)} files | rows: {sum(pq.ParquetFile(f).metadata.num_rows for f in fs):,}" if fs else "EMPTY")

In [ ]:
import subprocess, os, sys, textwrap, time, re, glob
from pathlib import Path
OMOP = os.environ["OMOP_DIR"]; OUT = os.environ["CEHR_GPT_DATA_DIR"]; os.makedirs(OUT, exist_ok=True)
subprocess.run(["bash","-lc", f"rm -rf {OMOP}/processed_measurement"])   # so --aggregate_by_hour reprocesses
print(f"input: {OMOP}\noutput: {OUT}")
assert glob.glob(f"{OMOP}/concept/*.parquet") and glob.glob(f"{OMOP}/qualified_concept_list/*.parquet")
assert glob.glob(f"{OMOP}/death/*.parquet"), "death not staged!"

LAUNCHER = "/home/user/repo/_cehr_launcher.py"
Path(LAUNCHER).write_text(textwrap.dedent(r'''
    import sys, importlib, runpy
    REAL = ["/home/user/repo/cehrbert_data-main/src","/home/user/repo/cehrbert-main","/home/user/repo/cehrgpt-main/src"]
    for p in reversed(REAL):
        while p in sys.path: sys.path.remove(p)
        sys.path.insert(0, p)
    for m in [m for m in list(sys.modules) if m=="cehrbert_data" or m.startswith("cehrbert_data.")]:
        del sys.modules[m]
    importlib.invalidate_caches()
    APP = "/home/user/repo/cehrbert_data-main/src/cehrbert_data/apps/generate_training_data.py"
    sys.argv = [APP] + sys.argv[1:]
    runpy.run_path(APP, run_name="__main__")
'''))
JVM_OPTS = ("--add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
            "--add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED "
            "--add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED")
LOG4J2_OPT = "-Dlog4j2.configurationFile=file:/home/user/repo/log4j2.properties"
PKG = "/home/user/repo/cehrbert_data-main/src:/home/user/repo/cehrbert-main:/home/user/repo/cehrgpt-main/src"
env = os.environ.copy()
env["JAVA_TOOL_OPTIONS"]=env["_JAVA_OPTIONS"]=JVM_OPTS; env["JAVA_HOME"]=sys.prefix
env["PATH"]=f"{sys.prefix}/bin:"+env.get("PATH",""); env["LD_LIBRARY_PATH"]=f"{sys.prefix}/lib:"+env.get("LD_LIBRARY_PATH","")
env["PYTHONPATH"]=PKG+":"+env.get("PYTHONPATH","")
os.makedirs("/home/user/repo/spark_tmp", exist_ok=True)
cmd = ["spark-submit","--master","local[8]","--driver-memory","48g",
       "--conf","spark.memory.fraction=0.75",
       "--conf","spark.sql.adaptive.enabled=true","--conf","spark.sql.adaptive.coalescePartitions.enabled=true",
       "--conf","spark.sql.adaptive.skewJoin.enabled=true",
       "--conf","spark.sql.shuffle.partitions=2000","--conf","spark.sql.files.maxPartitionBytes=64m",
       "--conf","spark.driver.maxResultSize=8g","--conf","spark.local.dir=/home/user/repo/spark_tmp",
       "--conf","spark.serializer=org.apache.spark.serializer.KryoSerializer",
       "--driver-java-options", f"{JVM_OPTS} {LOG4J2_OPT}", LAUNCHER,
       "--input_folder", OMOP, "--output_folder", OUT,
       "-d","1985-01-01","--att_type","day","--inpatient_att_type","day","-iv",
       "--include_concept_list","--gpt_patient_sequence","--is_new_patient_representation",
       "--include_inpatient_hour_token","--aggregate_by_hour",
       "--should_construct_artificial_visits","--disconnect_problem_list_records",
       "--domain_table_list","condition_occurrence","drug_exposure","procedure_occurrence","measurement"]
LOGDIR=Path("/home/user/repo/logs"); LOGDIR.mkdir(exist_ok=True)
log_path=LOGDIR/f"gen_train_full-{time.strftime('%Y%m%d-%H%M%S')}.log"
KEEP=re.compile(r'ERROR|Exception|Traceback|FAILED|py4j\.protocol|File "', re.I)
print(f"logfile: {log_path}\n"); n=0; t0=time.time()
with open(log_path,"w") as lf:
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
    for line in p.stdout:
        lf.write(line); n+=1
        if KEEP.search(line): print(line,end="")
        elif n%500==0: print(f"  ... {n} lines, {(time.time()-t0)/60:.1f} min",flush=True)
rc=p.wait(); print(f"\nrc={rc}  ({n} lines, {(time.time()-t0)/60:.1f} min, log -> {log_path})")
if rc!=0: print("".join(open(log_path).readlines()[-30:]))
else:
    import pyarrow.parquet as pq
    fs=glob.glob(f"{OUT}/patient_sequence/*.parquet")
    print(f"\npatient_sequence -> {len(fs)} files | rows: {sum(pq.ParquetFile(f).metadata.num_rows for f in fs):,}" if fs else "\n(no patient_sequence written?)")

In [ ]:
import glob, os
import pyarrow.parquet as pq
import polars as pl

OMOP = os.environ["OMOP_DIR"]
OUT  = os.environ["CEHR_GPT_DATA_DIR"]
def count(g):
    fs = glob.glob(g)
    return (len(fs), sum(pq.ParquetFile(f).metadata.num_rows for f in fs)) if fs else (None, 0)

print("=== OMOP input tables (metadata only) ===")
for t in ["person","visit_occurrence","condition_occurrence","drug_exposure",
          "procedure_occurrence","concept","concept_ancestor","concept_relationship", "death", "qualified_concept_list"]:
    nf, rows = count(f"{OMOP}/{t}/*.parquet")
    print(f"  {t:24s} " + ("MISSING" if nf is None else f"{rows:>14,} rows  ({nf} files)"))

print("\n=== patient_sequence (training input) ===")
nf, rows = count(f"{OUT}/patient_sequence/*.parquet")
if nf is None:
    print("  not generated yet")
else:
    f0 = glob.glob(f"{OUT}/patient_sequence/*.parquet")[0]
    print(f"  {rows:,} sequences across {nf} files")
    print(pl.read_parquet(f0, n_rows=1))


In [ ]:
import glob, pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc

seq = "/home/user/repo/cehrgpt_output_full/patient_sequence"
files = sorted(glob.glob(seq + "/*.parquet"))
FILL = {"ages": 0.0, "dates": 0, "epoch_times": 0}   # numeric cols cehrbert casts to int; null -> 0

def fill_nulls(col, val):
    arr = col.combine_chunks()
    if pa.types.is_list(arr.type):
        return pa.ListArray.from_arrays(arr.offsets, pc.fill_null(arr.values, val)).cast(arr.type)
    if pa.types.is_large_list(arr.type):
        return pa.LargeListArray.from_arrays(arr.offsets, pc.fill_null(arr.values, val)).cast(arr.type)
    return pc.fill_null(arr, val)

for fi, f in enumerate(files):
    t = pq.read_table(f)
    new = {n: (fill_nulls(t[n], FILL[n]) if n in FILL else t[n]) for n in t.column_names}
    pq.write_table(pa.table(new, schema=t.schema), f)
    if fi % 100 == 0: print(f"  ...{fi}/{len(files)}")
        
print("filled ages/dates/epoch_times nulls in all files")

In [ ]:
import glob, pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc
SRC="/home/user/repo/cehrgpt_output_full/patient_sequence"; kept=tot=0
for f in sorted(glob.glob(f"{SRC}/*.parquet")):
    t=pq.read_table(f); tot+=t.num_rows
    sub=t.filter(pc.less_equal(t["num_of_concepts"], 2048))
    pq.write_table(sub, f); kept+=sub.num_rows
print(f"re-filtered: kept {kept:,} of {tot:,} (<=2048 tokens, {100*kept/tot:.0f}%)")

In [ ]:
import glob, os, pyarrow as pa, pyarrow.parquet as pq
seq_dir="/home/user/repo/cehrgpt_output_full/patient_sequence"
files=sorted(glob.glob(f"{seq_dir}/*.parquet")); assert files
def L(t): return pa.list_(pa.field("element", t))
TARGET = pa.schema([
    ("cohort_member_id", pa.string()), ("person_id", pa.string()),
    ("concept_ids", L(pa.string())), ("visit_segments", L(pa.int32())),
    ("orders", L(pa.int32())), ("dates", L(pa.int32())), ("ages", L(pa.float64())),
    ("visit_concept_orders", L(pa.int32())), ("num_of_visits", pa.int32()),
    ("num_of_concepts", pa.int64()), ("concept_value_masks", L(pa.int32())),
    ("number_as_values", L(pa.float32())), ("concept_as_values", L(pa.string())),
    ("is_numeric_types", L(pa.int32())), ("mlm_skip_values", L(pa.int32())),
    ("priorities", L(pa.float64())), ("visit_concept_ids", L(pa.int32())),
    ("visit_rank_orders", L(pa.int32())), ("concept_orders", L(pa.int32())),
    ("record_ranks", L(pa.int32())), ("units", L(pa.string())), ("epoch_times", L(pa.int64())),
])
have=set(pq.read_schema(files[0]).names); want=set(TARGET.names)
missing, extra = want-have, have-want
if missing or extra:
    print(f"⚠ schema mismatch — missing {missing}, extra {extra}. Skipping cast; investigate before continuing.")
else:
    bad=0
    for f in files:
        try: pq.write_table(pq.read_table(f).cast(TARGET, safe=False), f)
        except Exception as e: bad+=1; print("cast skip", os.path.basename(f), repr(e)[:60])
    print(f"unified schema across {len(files)} files ({bad} skipped)")

In [ ]:
import glob, os, subprocess
import pyarrow.parquet as pq, pyarrow.compute as pc
from foundry.transforms import Dataset

NAME = "copd_patient_sequence_le2048"
SEQ  = "/home/user/repo/cehrgpt_output_full/patient_sequence"
try: Dataset.get(NAME).files()
except Exception as e: raise SystemExit(f"alias '{NAME}' not reachable ({repr(e)[:60]}) — create it + restart kernel")

files = sorted(glob.glob(f"{SEQ}/*.parquet")); assert files
rows = sum(pq.read_metadata(f).num_rows for f in files)
mx = max((pc.max(pq.read_table(f, columns=["num_of_concepts"])["num_of_concepts"]).as_py() or 0) for f in files[::10])
print(f"{len(files)} files, {rows:,} rows, sampled max {mx} tokens")
assert rows < 900_000 and mx <= 2048, "NOT the filtered set — run the ≤2048 re-filter FIRST"

TAR = "/home/user/repo/patient_sequence_le2048.tar"
subprocess.run(["bash","-lc", f"cd {SEQ} && tar cf {TAR} . && ls -lh {TAR}"], check=True)
Dataset.get(NAME).upload_file(TAR, "patient_sequence_le2048")
print(f"✓ banked filtered death+measurement patient_sequence ({rows:,} seqs) -> {NAME}")

In [ ]:
# ============== FIX → TEST → VERIFY (run once, ~3 min) ==============
patch_path = "/home/user/repo/cehrgpt-main/src/cehrgpt/gpt_utils.py"
def ensure(mark, body):
    s = open(patch_path).read()
    print("· already present:", mark) if mark in s else (open(patch_path,"a").write(body), print("· applied:", mark))

ensure("robustness patch: tolerate malformed time tokens", r'''

# robustness patch: tolerate malformed time tokens
import re as _re_patch
def extract_time_interval_in_days(token):  # noqa: F811
    try:
        return int(token.split("-")[1][1:])
    except (IndexError, ValueError):
        m = _re_patch.search(r"-?\d+$", str(token))
        return max(0, int(m.group())) if m else 0
''')
ensure("robustness patch v2: sanitize age/time sequences", '''

# robustness patch v2: sanitize age/time sequences
_orig_construct_age_sequence = construct_age_sequence
_orig_construct_time_sequence = construct_time_sequence
def construct_age_sequence(concept_ids, ages=None):  # noqa: F811
    out = _orig_construct_age_sequence(concept_ids, ages)
    return [0 if (a is None or a != a) else int(a) for a in out]
def construct_time_sequence(concept_ids, epoch_times=None):  # noqa: F811
    out = _orig_construct_time_sequence(concept_ids, epoch_times)
    return [0.0 if (t is None or t != t) else float(t) for t in out]
''')

# load the PATCHED functions fresh into this kernel
import sys, math, glob
for m in [x for x in sys.modules if x.startswith("cehrgpt")]: del sys.modules[m]
for p in ["/home/user/repo/cehrbert_data-main/src","/home/user/repo/cehrbert-main","/home/user/repo/cehrgpt-main/src"]:
    if p not in sys.path: sys.path.insert(0, p)
from cehrgpt.gpt_utils import construct_age_sequence, construct_time_sequence
import pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc

seq = "/home/user/repo/cehrgpt_output_full/patient_sequence"
files = sorted(glob.glob(seq + "/*.parquet")); assert files, "no patient_sequence files"
def isbad(x): return x is None or (isinstance(x, float) and math.isnan(x))

# (A) FUNCTIONAL: run the PATCHED reconstruction on real records + the exact int32 array op
st = pq.read_table(files[0], columns=["concept_ids","ages","epoch_times"])
ci, ag, et = st["concept_ids"].to_pylist(), st["ages"].to_pylist(), st["epoch_times"].to_pylist()
fn_fail = 0
for c,a,e in zip(ci, ag, et):
    aa, tt = construct_age_sequence(c,a), construct_time_sequence(c,e)
    if any(isbad(x) for x in aa) or any(isbad(x) for x in tt): fn_fail += 1; continue
    try: pa.array(aa, type=pa.int32())          # the exact op that crashed
    except Exception: fn_fail += 1
print(f"(A) patched function on {len(ci):,} real records -> {fn_fail} bad (want 0)")

# (B) FULL SCAN: prove the sanitization clears every file and the int cast succeeds
raw_bad = resid = cast_fail = 0
for fi, f in enumerate(files):
    t = pq.read_table(f, columns=["ages","epoch_times"])
    for name, tgt in [("ages", pa.int32()), ("epoch_times", pa.int64())]:
        v = pc.list_flatten(t[name].combine_chunks())
        raw_bad += (pc.sum(pc.is_null(v)).as_py() or 0)
        if pa.types.is_floating(v.type): raw_bad += (pc.sum(pc.is_nan(v)).as_py() or 0)
        s = pc.if_else(pc.is_null(v), 0, v)                                   # null -> 0 (what the patch does)
        if pa.types.is_floating(s.type): s = pc.if_else(pc.is_nan(s), 0, s)   # nan  -> 0
        resid += (pc.sum(pc.is_null(s)).as_py() or 0)
        if pa.types.is_floating(s.type): resid += (pc.sum(pc.is_nan(s)).as_py() or 0)
        try: s.cast(tgt)
        except Exception as ex: cast_fail += 1; print("CAST FAIL", f, name, repr(ex)[:70])
    if fi % 150 == 0: print(f"  ...scanned {fi}/{len(files)} files")

print("\n================ VERDICT ================")
print(f"bad age/time values in raw data (the bug):  {raw_bad:,}")
print(f"(A) patched-function bad outputs:           {fn_fail}   (must be 0)")
print(f"(B) residual NaN/null after patch:          {resid}   (must be 0)")
print(f"(B) files failing the int cast:             {cast_fail}   (must be 0)")
ok = (fn_fail == 0 and resid == 0 and cast_fail == 0)
print(f"\n✅ CORRECT — NaN→int32 crash is neutralized across all {len(files)} files. Safe to launch."
      if ok else "\n❌ NOT SAFE — do NOT train; see the CAST FAIL lines above.")

In [ ]:
import glob, os, subprocess, pyarrow as pa, pyarrow.parquet as pq
from foundry.transforms import Dataset

NAME = "copd_patient_sequence_full"            # WRITABLE Foundry dataset (create the alias in the UI first)
seq_dir = f"{os.environ['CEHR_GPT_DATA_DIR']}/patient_sequence"
files = sorted(glob.glob(f"{seq_dir}/*.parquet"))
assert files, "no patient_sequence parquet found"

def L(t): return pa.list_(pa.field("element", t))   # list child named 'element' to match the files
TARGET = pa.schema([
    ("cohort_member_id", pa.string()), ("person_id", pa.string()),
    ("concept_ids", L(pa.string())), ("visit_segments", L(pa.int32())),
    ("orders", L(pa.int32())), ("dates", L(pa.int32())), ("ages", L(pa.float64())),
    ("visit_concept_orders", L(pa.int32())), ("num_of_visits", pa.int32()),
    ("num_of_concepts", pa.int64()), ("concept_value_masks", L(pa.int32())),
    ("number_as_values", L(pa.float32())), ("concept_as_values", L(pa.string())),
    ("is_numeric_types", L(pa.int32())), ("mlm_skip_values", L(pa.int32())),
    ("priorities", L(pa.float64())), ("visit_concept_ids", L(pa.int32())),
    ("visit_rank_orders", L(pa.int32())), ("concept_orders", L(pa.int32())),
    ("record_ranks", L(pa.int32())), ("units", L(pa.string())), ("epoch_times", L(pa.int64())),
])

# 1) force every file to the same schema (fixes int/float drift; makes the training reads consistent)
for f in files:
    try:
        pq.write_table(pq.read_table(f).cast(TARGET, safe=False), f)
    except Exception as e:
        print("CAST FAILED on", os.path.basename(f), "->", e); raise
print(f"unified schema across {len(files)} files")


In [ ]:
# 2) tar (file-level) + upload_file — do NOT concat+write_table; that OOMs the 65 GB pod at 1.24M rows
TAR = "/home/user/repo/patient_sequence_full.tar"
subprocess.run(["bash","-lc", f"cd {seq_dir} && tar cf {TAR} . && ls -lh {TAR}"], check=True)
Dataset.get(NAME).upload_file(TAR, "patient_sequence_full")
print(f"OK — cached patient_sequence to '{NAME}'")


### Restore after a wipe
Run the Environment cells first, then this restore (patient_sequence + latest checkpoint + tokenizer from Foundry), then the training cell with `RESUME = True`.


In [ ]:
import os, glob, shutil, subprocess
from pathlib import Path
from foundry.transforms import Dataset
import pyarrow.parquet as pq

os.environ["OMOP_DIR"]="/home/user/repo/omop_synthea"
os.environ["CEHR_GPT_DATA_DIR"]="/home/user/repo/cehrgpt_output_full"
os.environ["CEHR_GPT_MODEL_DIR"]="/home/user/repo/model_run_full"
DATA=os.environ["CEHR_GPT_DATA_DIR"]; MODEL=os.environ["CEHR_GPT_MODEL_DIR"]; OUT=MODEL+"_real"

# 1) filtered death+measurement patient_sequence
ps=Path(f"{DATA}/patient_sequence"); subprocess.run(["bash","-lc",f"rm -rf {ps} && mkdir -p {ps}"],check=True)
tar=max(Dataset.get("copd_patient_sequence_le2048").files().download().values(), key=lambda p:os.path.getsize(p))
subprocess.run(["bash","-lc",f"tar xf {tar} -C {ps}"],check=True)
fs=glob.glob(f"{ps}/*.parquet")
print(f"patient_sequence: {len(fs)} files, {sum(pq.ParquetFile(x).metadata.num_rows for x in fs):,} seqs (filtered)")

# 2) clean slate — empty model dir, no stale prep/checkpoints, NO v1 tokenizer
shutil.rmtree(OUT, ignore_errors=True)
for d in glob.glob(f"{DATA}/dataset_prepared*"): shutil.rmtree(d, ignore_errors=True)
Path(MODEL).mkdir(parents=True, exist_ok=True)
for p in glob.glob(f"{MODEL}/*"): shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)

# 3) resume the v2 model ONLY if a v2 checkpoint was banked overnight
resume=False
try:
    tars=[p for p in Dataset.get("copd_cehrgpt_v2_resume").files().download().values() if str(p).endswith(".tar")]
    if tars:
        st=Path("/home/user/repo/_v2_stage"); subprocess.run(["bash","-lc",f"rm -rf {st} && mkdir -p {st}"],check=True)
        subprocess.run(["bash","-lc",f"tar xf {max(tars,key=os.path.getsize)} -C {st}"],check=True)
        tok=next((d for d,_,fn in os.walk(st) if "cehrgpt_tokenizer.json" in fn), None)
        if tok: subprocess.run(["bash","-lc",f"cp -a {tok}/. {MODEL}/"],check=True)
        Path(OUT).mkdir(parents=True, exist_ok=True)
        for ck in glob.glob(f"{st}/**/checkpoint-*", recursive=True):
            subprocess.run(["bash","-lc",f"cp -a {ck} {OUT}/"],check=True)
        resume=bool(glob.glob(f"{OUT}/checkpoint-*"))
except Exception as e:
    print("no v2 bank:", repr(e)[:80])
cks=[os.path.basename(c) for c in glob.glob(f"{OUT}/checkpoint-*")]
print(f"\nmodel_run_full: {os.listdir(MODEL) or 'empty → fresh tokenizer'} | v2 checkpoints: {cks or 'none'}")
print(f"→ set RESUME={'True' if resume else 'False'} in the training cell")

In [ ]:
import os, glob, shutil, subprocess
from pathlib import Path
from foundry.transforms import Dataset
import pyarrow.parquet as pq

os.environ["OMOP_DIR"]="/home/user/repo/omop_synthea"
os.environ["CEHR_GPT_DATA_DIR"]="/home/user/repo/cehrgpt_output_full"
os.environ["CEHR_GPT_MODEL_DIR"]="/home/user/repo/model_run_full"
DATA=os.environ["CEHR_GPT_DATA_DIR"]; MODEL=os.environ["CEHR_GPT_MODEL_DIR"]; OUT=MODEL+"_real"

# 1) FILTERED patient_sequence — comes back already <=2048, no re-filter needed
ps=Path(f"{DATA}/patient_sequence"); subprocess.run(["bash","-lc",f"rm -rf {ps} && mkdir -p {ps}"],check=True)
tar=max(Dataset.get("copd_patient_sequence_le2048").files().download().values(), key=lambda p:os.path.getsize(p))
subprocess.run(["bash","-lc",f"tar xf {tar} -C {ps}"],check=True)
fs=glob.glob(f"{ps}/*.parquet")
print(f"patient_sequence: {len(fs)} files, {sum(pq.ParquetFile(x).metadata.num_rows for x in fs):,} seqs (filtered)")

# 2) start clean — empty model dir, no stale prep/checkpoints (NO v1 tokenizer!)
shutil.rmtree(OUT, ignore_errors=True)
for d in glob.glob(f"{DATA}/dataset_prepared*"): shutil.rmtree(d, ignore_errors=True)
Path(MODEL).mkdir(parents=True, exist_ok=True)
for p in glob.glob(f"{MODEL}/*"): shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)

# 3) resume the v2 model ONLY if a v2 checkpoint was banked (2nd+ sessions)
resume=False
try:
    tars=[p for p in Dataset.get("copd_cehrgpt_v2_resume").files().download().values() if str(p).endswith(".tar")]
    if tars:
        st=Path("/home/user/repo/_v2_stage"); subprocess.run(["bash","-lc",f"rm -rf {st} && mkdir -p {st}"],check=True)
        subprocess.run(["bash","-lc",f"tar xf {max(tars,key=os.path.getsize)} -C {st}"],check=True)
        tok=next((d for d,_,fn in os.walk(st) if "cehrgpt_tokenizer.json" in fn), None)
        if tok: subprocess.run(["bash","-lc",f"cp -a {tok}/. {MODEL}/"],check=True)
        Path(OUT).mkdir(parents=True, exist_ok=True)
        for ck in glob.glob(f"{st}/**/checkpoint-*", recursive=True):
            subprocess.run(["bash","-lc",f"cp -a {ck} {OUT}/"],check=True)
        resume=bool(glob.glob(f"{OUT}/checkpoint-*"))
except Exception as e:
    print("no v2 bank yet:", repr(e)[:80])
print(f"model_run_full: {os.listdir(MODEL) or 'empty → fresh tokenizer builds'}")
print(f"→ set RESUME={'True' if resume else 'False'} in the training cell")

## Step 2 — Train CEHR-GPT (full, resumable across 6-hour sessions)
`save_strategy=steps` so a 6-hour session always banks checkpoints (a full-data epoch is ~10h). Run the GPU-cleanup cell first if you hit a CUDA OOM.


In [ ]:
import os, gc, sys, subprocess, signal, time

def gpu_status(label):
    print(f"\n=== GPU {label} ===")
    subprocess.run(["nvidia-smi","--query-gpu=memory.used,memory.free,memory.total","--format=csv"])
    subprocess.run(["nvidia-smi","--query-compute-apps=pid,used_memory,process_name","--format=csv"])

gpu_status("before")

# 1) kill orphaned CEHR-GPT training subprocesses still holding VRAM (never this kernel)
my_pid = os.getpid()
pids = subprocess.run(["nvidia-smi","--query-compute-apps=pid","--format=csv,noheader"],
                      capture_output=True, text=True).stdout.split()
killed = []
for tok in pids:
    if not tok.strip().isdigit(): continue
    pid = int(tok)
    if pid == my_pid: continue                          # protect the notebook kernel
    try: name = open(f"/proc/{pid}/cmdline").read().replace("\x00"," ")
    except Exception: name = ""
    if "cehrgpt" in name.lower():                        # only our training runner
        try: os.kill(pid, signal.SIGKILL); killed.append(pid)
        except Exception as e: print("  couldn't kill", pid, e)
subprocess.run(["bash","-c","pkill -9 -f hf_cehrgpt_pretrain_runner 2>/dev/null; true"])
print("killed stray training procs:", killed or "none")

# 2) clear torch cache ONLY if this kernel already holds a CUDA context (don't create one on a fresh kernel)
if "torch" in sys.modules:
    for v in ["x","y","model","trainer","batch","out","logits"]:
        globals().pop(v, None)
    gc.collect()
    t = sys.modules["torch"]
    if t.cuda.is_available():
        t.cuda.empty_cache(); t.cuda.ipc_collect(); t.cuda.reset_peak_memory_stats()
        print(f"cleared kernel torch cache (reserved now {t.cuda.memory_reserved()/1e9:.2f} GB)")
else:
    print("torch not imported here — no kernel CUDA context to clear (ideal)")

time.sleep(2)
gpu_status("after")

In [ ]:
import subprocess, sys, os, time, textwrap
from pathlib import Path
WATCHER = "/home/user/repo/resume_watcher_v2.py"
Path("/home/user/repo/logs").mkdir(parents=True, exist_ok=True)
Path(WATCHER).write_text(textwrap.dedent('''
    import time, glob, os, subprocess
    from foundry.transforms import Dataset
    OUTPUT_DIR = "/home/user/repo/model_run_full_real"
    TOKENIZER  = "/home/user/repo/model_run_full"
    PREP       = "/home/user/repo/cehrgpt_output_full/dataset_prepared_real"
    CKPT_DS, PREP_DS = "copd_cehrgpt_v2_resume", "copd_dataset_prepared_v2"
    INTERVAL = 120
    def complete(c):
        return (os.path.exists(c+"/trainer_state.json") and
                (os.path.exists(c+"/model.safetensors") or os.path.exists(c+"/pytorch_model.bin")))
    def latest():
        cks=[c for c in glob.glob(OUTPUT_DIR+"/checkpoint-*") if complete(c)]
        return max(cks, key=lambda c:int(c.split("-")[-1])) if cks else None
    last=None; prep_banked=False
    print(time.strftime("%H:%M:%S"),"v2 watcher up ->",CKPT_DS,flush=True)
    while True:
        if not prep_banked and glob.glob(PREP+"/**/dataset_info.json", recursive=True):
            try:
                pt="/home/user/repo/dataset_prepared_v2.tar"
                subprocess.run(["bash","-lc",f"cd {os.path.dirname(PREP)} && tar cf {pt} dataset_prepared_real"],check=True)
                Dataset.get(PREP_DS).upload_file(pt,"dataset_prepared_v2.tar")
                prep_banked=True; print(time.strftime("%H:%M:%S"),"BANKED prep",flush=True)
            except Exception as e:
                print(time.strftime("%H:%M:%S"),"prep bank skipped:",repr(e)[:120],flush=True); prep_banked=True
        try:
            ck=latest()
            if ck and ck!=last:
                t="/home/user/repo/resume.tar"; n=os.path.basename(ck)
                subprocess.run(["bash","-lc",f"tar cf {t} -C {os.path.dirname(TOKENIZER)} {os.path.basename(TOKENIZER)} -C {OUTPUT_DIR} {n}"],check=True)
                Dataset.get(CKPT_DS).upload_file(t,"resume.tar")
                last=ck; print(time.strftime("%H:%M:%S"),"BANKED",n,f"({os.path.getsize(t)/1e9:.2f} GB)",flush=True)
        except Exception as e:
            print(time.strftime("%H:%M:%S"),"ckpt bank error:",repr(e)[:120],flush=True)
        time.sleep(INTERVAL)
    '''))
subprocess.run(["bash","-lc","pkill -f resume_watcher_v2.py 2>/dev/null; true"]); time.sleep(1)
logf=open("/home/user/repo/logs/watcher_v2.log","a")
proc=subprocess.Popen([sys.executable,"-u",WATCHER],stdout=logf,stderr=subprocess.STDOUT,start_new_session=True,env=os.environ.copy())
print("launched PID",proc.pid); time.sleep(6)
print(subprocess.run(["bash","-lc","tail -8 /home/user/repo/logs/watcher_v2.log"],capture_output=True,text=True).stdout)

In [ ]:
from foundry.transforms import Dataset
try: Dataset.get("copd_cehrgpt_v2_resume").files(); print("✓ copd_cehrgpt_v2_resume ready")
except Exception as e: print("✗ create it in the UI + restart kernel:", repr(e)[:80])

In [ ]:
# robustness patch: let the tokenizer survive malformed/negative time tokens like "i-D-6"
patch_path = "/home/user/repo/cehrgpt-main/src/cehrgpt/gpt_utils.py"
MARK = "robustness patch: tolerate malformed time tokens"
override = f'''

# {MARK}  (overrides extract_time_interval_in_days for tokens like "i-D-6")
import re as _re_patch
def extract_time_interval_in_days(token):  # noqa: F811
    try:
        return int(token.split("-")[1][1:])            # original fast path: "i-D6" -> 6
    except (IndexError, ValueError):
        m = _re_patch.search(r"-?\\d+$", str(token))     # trailing signed int: "i-D-6" -> -6
        return max(0, int(m.group())) if m else 0        # clamp negatives/unparseable to 0
'''
src = open(patch_path).read()
if MARK not in src:
    with open(patch_path, "a") as f: f.write(override)
    print("patched:", patch_path)
else:
    print("already patched")

In [ ]:
# ===================== PRE-FLIGHT — run RIGHT before training =====================
import os, sys, glob, shutil, subprocess
from pathlib import Path

REPO = "/home/user/repo"
DATA, MODEL = f"{REPO}/cehrgpt_output_full", f"{REPO}/model_run_full"
PKG = [f"{REPO}/cehrbert-main", f"{REPO}/cehrgpt-main/src", f"{REPO}/cehrbert_data-main/src"]
checks = []
def chk(name, ok, detail=""):
    checks.append(ok); print(f"  {'✅' if ok else '❌'} {name:40s} {detail}")

print("=== 1) GPU ===")
try:
    import torch; g = torch.cuda.is_available()
    chk("CUDA GPU available", g, torch.cuda.get_device_name(0) if g else "FALSE → CPU = months, STOP")
except Exception as e: chk("torch import", False, repr(e)[:60])

print("=== 2) both gpt_utils patches on disk ===")
gu = open(f"{REPO}/cehrgpt-main/src/cehrgpt/gpt_utils.py").read()
chk("patch v1 (time tokens)", "robustness patch: tolerate malformed time tokens" in gu)
chk("patch v2 (age/time sanitize)", "robustness patch v2: sanitize age/time sequences" in gu)

print("=== 3) patch is LIVE in a training-style subprocess ===")
probe = f"""
import sys; sys.path[:0] = {PKG!r}
from cehrgpt.gpt_utils import construct_age_sequence, construct_time_sequence
import pyarrow as pa
a = construct_age_sequence(['Year:2000','Age:40','x'], [None, float('nan'), 41.0])
t = construct_time_sequence(['Year:2000','Age:40','x'], [None, float('nan'), 1.0e9])
bad = [x for x in a + t if x is None or x != x]
pa.array(a, type=pa.int32())                      # the exact op that crashed
print('SUBPROC_OK', a, 'no-nan' if not bad else 'BAD')
"""
r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True,
                   env={**os.environ, "PYTHONPATH": ":".join(PKG)})
ok3 = "SUBPROC_OK" in r.stdout and "no-nan" in r.stdout
chk("subprocess runs PATCHED funcs, no NaN", ok3,
    (r.stdout.strip().splitlines()[-1] if r.stdout else r.stderr.strip()[-80:]))

print("=== 4) data + dirs ===")
ps = glob.glob(f"{DATA}/patient_sequence/*.parquet")
chk("patient_sequence files present", len(ps) >= 500, f"{len(ps)} files")
chk("model_run_full exists", Path(MODEL).exists(), MODEL)
prep = Path(f"{DATA}/dataset_prepared_real")
stale = prep.exists() and any(prep.rglob("*"))
chk("no stale dataset_prepared_real", not stale, "exists → set REBUILD_PREP=True" if stale else "clean")

print("=== 5) import chain (subprocess) ===")
r2 = subprocess.run([sys.executable, "-c",
   "import cehrbert_data, cehrbert, cehrgpt, cehrgpt.runners.hf_cehrgpt_pretrain_runner; print('IMPORTS_OK')"],
   capture_output=True, text=True, env={**os.environ, "PYTHONPATH": ":".join(PKG)})
chk("cehrgpt runner imports", "IMPORTS_OK" in r2.stdout, "" if "IMPORTS_OK" in r2.stdout else r2.stderr.strip()[-80:])

print("=== 6) checkpoint safety net ===")
w = subprocess.run(["bash","-lc","pgrep -af resume_watcher.py"], capture_output=True, text=True).stdout.strip()
chk("watcher process alive", bool(w), w.split(chr(10))[0] if w else "NOT running → a crash loses progress")
try:
    from foundry.transforms import Dataset
    Dataset.get("copd_cehrgpt_full_resume").files().download()
    chk("resume alias reachable", True, "copd_cehrgpt_full_resume")
except Exception as e:
    chk("resume alias reachable", "AliasNotDefined" not in str(e), repr(e)[:50])

print("=== 7) disk headroom ===")
free = shutil.disk_usage(REPO)[2] / 1e9
chk("free disk > 30 GB", free > 30, f"{free:.0f} GB free")

print("\n================ GO / NO-GO ================")
print("✅ ALL CLEAR — launch the training cell now." if all(checks)
      else "❌ STOP — resolve the ❌ line(s) above before launching.")

In [ ]:
import datasets, os, sys, subprocess
tbl = os.path.join(os.path.dirname(datasets.__file__), "table.py")
MARK = "NaN-safe integer cast patch"
if MARK not in open(tbl).read():
    open(tbl, "a").write('''

# === NaN-safe integer cast patch ===
import pyarrow as _pa_safe, pyarrow.compute as _pc_safe
_orig_array_cast_safe = array_cast
def array_cast(array, pa_type, *args, **kwargs):  # noqa: F811
    try:
        if _pa_safe.types.is_integer(pa_type) and _pa_safe.types.is_floating(array.type):
            m = _pc_safe.or_(_pc_safe.is_null(array), _pc_safe.is_nan(array))
            array = _pc_safe.if_else(m, _pa_safe.scalar(0, type=array.type), array)
    except Exception:
        pass
    return _orig_array_cast_safe(array, pa_type, *args, **kwargs)
''')
    print("patched", tbl)
else:
    print("already patched")

# verify in a fresh subprocess (exactly what the workers load)
probe = """
from datasets.table import array_cast, cast_array_to_feature
from datasets.features import Sequence, Value
import pyarrow as pa
leaf = array_cast(pa.array([1.0, float('nan'), 3.0]), pa.int32()).to_pylist()
try: nested = cast_array_to_feature(pa.array([[1.0,float('nan')],[None,4.0]]), Sequence(Value('int32'))).to_pylist()
except Exception as e: nested = 'ERR:'+repr(e)[:40]
assert leaf == [1,0,3], leaf
print('CAST_SAFE_OK leaf=', leaf, 'nested=', nested)
"""
r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
print(r.stdout.strip() or ("FAILED:\n" + r.stderr[-600:]))

In [ ]:
# SPEED FIX: memoize get_vocab so it's built once per worker, not rebuilt per record (the ~30 ex/s bottleneck)
tok = "/home/user/repo/cehrgpt-main/src/cehrgpt/models/tokenization_hf_cehrgpt.py"
if "speed patch: memoize get_vocab" not in open(tok).read():
    open(tok, "a").write('''

# speed patch: memoize get_vocab (vocab is immutable during prep; avoids rebuilding it per record)
def _cehrgpt_get_vocab_cached(self):
    v = getattr(self, "_vocab_cache", None)
    if v is None:
        v = self._tokenizer.get_vocab(); self._vocab_cache = v
    return v
CehrGptTokenizer.get_vocab = _cehrgpt_get_vocab_cached
''')
    print("patched get_vocab (memoized)")
else:
    print("get_vocab already memoized")

In [ ]:
import os, sys, glob, shutil, subprocess, time
from pathlib import Path

allf = sorted(glob.glob("/home/user/repo/cehrgpt_output_full/patient_sequence/*.parquet"))
subset = allf[::75][:8]                       # 8 files spread across the cohort (likely to include a bad record)
SUB = "/home/user/repo/_smoke"; shutil.rmtree(SUB, ignore_errors=True)
Path(f"{SUB}/patient_sequence").mkdir(parents=True); Path(f"{SUB}/model").mkdir()
for f in subset: shutil.copy(f, f"{SUB}/patient_sequence/")

env = os.environ.copy()
env.update(TOKENIZERS_PARALLELISM="false", WANDB_DISABLED="true", TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD="1")
env["LD_LIBRARY_PATH"] = f"{sys.prefix}/lib:" + env.get("LD_LIBRARY_PATH","")
env["PYTHONPATH"] = ":".join(["/home/user/repo/cehrbert-main","/home/user/repo/cehrgpt-main/src","/home/user/repo/cehrbert_data-main/src", env.get("PYTHONPATH","")])

cmd = [sys.executable,"-u","-m","cehrgpt.runners.hf_cehrgpt_pretrain_runner",
  "--model_name_or_path", f"{SUB}/model","--tokenizer_name_or_path", f"{SUB}/model","--output_dir", f"{SUB}/out",
  "--data_folder", f"{SUB}/patient_sequence","--dataset_prepared_path", f"{SUB}/prep","--preprocessing_num_workers","4",
  "--do_train","true","--seed","42","--hidden_size","768","--num_hidden_layers","12","--max_position_embeddings","1024",
  "--sample_packing","--max_tokens_per_batch","4096","--max_steps","2","--num_train_epochs","1",
  "--learning_rate","0.0002","--save_strategy","no","--evaluation_strategy","no", "--load_best_model_at_end","false", "--use_early_stopping","false","--logging_steps","1",
  "--report_to","none","--bf16","true","--dataloader_num_workers","0"]

print("SMOKE: real prep + 2 training steps on 8 files...")
t0=time.time(); r=subprocess.run(cmd, capture_output=True, text=True, env=env); dur=(time.time()-t0)/60
print("\n".join((r.stdout+r.stderr).strip().splitlines()[-12:]))
print(f"\nrc={r.returncode}  ({dur:.1f} min)")
print("\n✅ SMOKE PASSED — full prep+train path completes. NOW safe to launch the real run."
      if r.returncode==0 else "\n❌ SMOKE FAILED — see the output above; do NOT launch the full run.")

In [ ]:
import glob, numpy as np, pyarrow.parquet as pq
files=sorted(glob.glob("/home/user/repo/cehrgpt_output_full/patient_sequence/*.parquet"))
lens=[]
for f in files[::20]:
    lens.extend(pq.read_table(f, columns=["num_of_concepts"])["num_of_concepts"].to_pylist())
a=np.array([x for x in lens if x], dtype=float)
print(f"sampled {len(a):,} sequences")
for p in [50,90,95,99,99.9]: print(f"  p{p:<4}: {np.percentile(a,p):>8,.0f} tokens")
print(f"  max : {a.max():>8,.0f}")
for thr in [1024,2048,4096,8192]:
    print(f"  > {thr:>5}: {(a>thr).sum():>7,}  ({100*(a>thr).mean():.2f}%)")

In [ ]:
import glob, os, pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc
SRC="/home/user/repo/cehrgpt_output_full/patient_sequence"; MAXLEN=2048
kept=tot=0
for f in sorted(glob.glob(f"{SRC}/*.parquet")):
    t=pq.read_table(f); tot+=t.num_rows
    pq.write_table(t.filter(pc.less_equal(t["num_of_concepts"], MAXLEN)), f)
    kept+=t.filter(pc.less_equal(t["num_of_concepts"], MAXLEN)).num_rows
print(f"kept {kept:,} of {tot:,} sequences (<= {MAXLEN} tokens, {100*kept/tot:.0f}%)")

In [ ]:
import glob, pyarrow.parquet as pq, numpy as np
files = sorted(glob.glob("/home/user/repo/cehrgpt_output_full/patient_sequence/*.parquet"))[::6]
lens=[]
for f in files:
    lens.extend(pq.read_table(f, columns=["num_of_concepts"])["num_of_concepts"].to_pylist())

a=np.array([x for x in lens if x], dtype=float)
print(f"sampled {len(a):,} sequences from {len(files)} files")
for p in [50,90,99,99.9,99.99]: print(f"  p{p:<6}: {np.percentile(a,p):>9.0f} tokens")
print(f"  max: {a.max():.0f}")
for thr in [2048,4096,8192,16384]:
    print(f"  > {thr:>6}: {(a>thr).sum():>7,}  ({100*(a>thr).mean():.3f}%)")

# also confirm the memoize patch is actually on disk this session:
print("\nget_vocab memoized:", "speed patch: memoize get_vocab" in open("/home/user/repo/cehrgpt-main/src/cehrgpt/models/tokenization_hf_cehrgpt.py").read())

In [ ]:
import glob, os, shutil, pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc
MODEL="/home/user/repo/model_run_full"; DATA="/home/user/repo/cehrgpt_output_full"
# 1) clear the OLD v1 tokenizer so a fresh measurement tokenizer builds
for p in glob.glob(f"{MODEL}/*"): shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)
os.makedirs(MODEL, exist_ok=True); print("model_run_full:", os.listdir(MODEL) or "empty ✓")
# 2) re-filter patient_sequence to <=2048 in case the restored set is unfiltered
SRC=f"{DATA}/patient_sequence"; kept=tot=0
for f in sorted(glob.glob(f"{SRC}/*.parquet")):
    t=pq.read_table(f); tot+=t.num_rows
    sub=t.filter(pc.less_equal(t["num_of_concepts"], 2048)); pq.write_table(sub, f); kept+=sub.num_rows
print(f"re-filtered: kept {kept:,} of {tot:,}  (expected ≈772k for the full cohort)")

In [ ]:
import glob, os, subprocess
import pyarrow.parquet as pq, pyarrow.compute as pc
from foundry.transforms import Dataset

NAME = "copd_patient_sequence_le2048"   # create in the UI; restart kernel if newly created so the SDK sees it
SEQ  = "/home/user/repo/cehrgpt_output_full/patient_sequence"

try: Dataset.get(NAME).files()
except Exception as e: raise SystemExit(f"alias '{NAME}' not reachable ({repr(e)[:60]}) — create it + restart kernel")

files = sorted(glob.glob(f"{SEQ}/*.parquet")); assert files, "no patient_sequence parquet"
rows = sum(pq.read_metadata(f).num_rows for f in files)
mx = max((pc.max(pq.read_table(f, columns=["num_of_concepts"])["num_of_concepts"]).as_py() or 0) for f in files[::10])
print(f"{len(files)} files, {rows:,} rows, sampled max {mx} tokens")
assert rows < 900_000 and mx <= 2048, "NOT the filtered set — run the ≤2048 re-filter FIRST, then bank"

TAR = "/home/user/repo/patient_sequence_le2048.tar"
subprocess.run(["bash","-lc", f"cd {SEQ} && tar cf {TAR} . && ls -lh {TAR}"], check=True)
Dataset.get(NAME).upload_file(TAR, "patient_sequence_le2048")
print(f"✓ banked filtered patient_sequence ({rows:,} seqs) -> {NAME}")

In [ ]:
import glob, os, subprocess
from foundry.transforms import Dataset

print("=== LOCAL  ===")
print("  checkpoints:", [os.path.basename(c) for c in glob.glob("/home/user/repo/model_run_full_real/checkpoint-*")] or "none")
print("  prep built :", bool(glob.glob("/home/user/repo/cehrgpt_output_full/dataset_prepared_real/**/dataset_info.json", recursive=True)))
print("  tokenizer  :", os.path.exists("/home/user/repo/model_run_full/cehrgpt_tokenizer.json"))

print("\n=== Foundry resume banks — did overnight save anything? ===")
for name in ["copd_cehrgpt_v2_resume","copd_cehrgpt_full_resume","copd_dataset_prepared_v2"]:
    try:
        f=Dataset.get(name).files().download()
        tars=[p for p in f.values() if str(p).endswith(".tar")]
        gb=sum(os.path.getsize(p) for p in f.values())/1e9
        cks=subprocess.run(["bash","-lc",f"tar tf {max(tars,key=os.path.getsize)} 2>/dev/null | grep -oE 'checkpoint-[0-9]+' | sort -u | tr '\\n' ' '"],capture_output=True,text=True).stdout.strip() if tars else ""
        print(f"  {name}: {gb:.2f} GB | checkpoints inside: {cks or '(none)'}")
    except Exception as e:
        print(f"  {name}: not defined/empty ({repr(e)[:45]})")

In [ ]:
from foundry.transforms import Dataset
for n in ["copd_cehrgpt_v2_resume","copd_dataset_prepared_v2"]:
   try: Dataset.get(n).files(); print("✓", n)
   except Exception as e: print("✗", n, repr(e)[:60])

In [ ]:
import os, sys, shutil, re, time, subprocess, torch
from pathlib import Path

NUM_EPOCHS, MODEL_CTX_LEN, GPU_MAX_TOKENS, SAVE_STEPS = 1, 1024, 16384, 500
RESUME, REBUILD_PREP, FRESH_OUTPUT = False, True, True
if RESUME: FRESH_OUTPUT = False

OMOP_DIR="/home/user/repo/omop_synthea"; CEHR_GPT_DATA_DIR="/home/user/repo/cehrgpt_output_full"; CEHR_GPT_MODEL_DIR="/home/user/repo/model_run_full"
os.environ.update(OMOP_DIR=OMOP_DIR, CEHR_GPT_DATA_DIR=CEHR_GPT_DATA_DIR, CEHR_GPT_MODEL_DIR=CEHR_GPT_MODEL_DIR)
MODEL_DIR_IN, DATA_DIR = CEHR_GPT_MODEL_DIR, CEHR_GPT_DATA_DIR
DATASET_PREP = f"{DATA_DIR}/dataset_prepared_real"
if REBUILD_PREP and Path(DATASET_PREP).exists(): shutil.rmtree(DATASET_PREP)
for label, pth in [("model/tokenizer dir", MODEL_DIR_IN), ("patient_sequence", f"{DATA_DIR}/patient_sequence")]:
    if not Path(pth).exists(): raise FileNotFoundError(f"[path check] {label} not found: {pth}")

HAS_GPU = torch.cuda.is_available()
print(f"GPU available: {HAS_GPU}" + (f"  ({torch.cuda.get_device_name(0)})" if HAS_GPU else "  -- CPU"))
precision = ["--bf16","true","--fp16","false"] if (HAS_GPU and torch.cuda.is_bf16_supported()) else (["--bf16","false","--fp16","true"] if HAS_GPU else ["--bf16","false","--fp16","false"])
MAX_TOK = str(GPU_MAX_TOKENS) if HAS_GPU else str(MODEL_CTX_LEN)

env = os.environ.copy()
env["OMP_NUM_THREADS"]="8" if HAS_GPU else "16"
env["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"; env["TOKENIZERS_PARALLELISM"]="false"
env["WANDB_DISABLED"]="true"; env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"]="1"
if not HAS_GPU: env["CUDA_VISIBLE_DEVICES"]=""
env["LD_LIBRARY_PATH"]=f"{sys.prefix}/lib:"+env.get("LD_LIBRARY_PATH","")
env["PYTHONPATH"]=":".join(["/home/user/repo/cehrbert-main","/home/user/repo/cehrgpt-main/src","/home/user/repo/cehrbert_data-main/src",env.get("PYTHONPATH","")])

OUTPUT_DIR = MODEL_DIR_IN.rstrip("/") + "_real"
if FRESH_OUTPUT and Path(OUTPUT_DIR).exists(): shutil.rmtree(OUTPUT_DIR)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True); print("output dir:", OUTPUT_DIR)

cmd = [sys.executable,"-u","-m","cehrgpt.runners.hf_cehrgpt_pretrain_runner",
   "--model_name_or_path",MODEL_DIR_IN,"--tokenizer_name_or_path",MODEL_DIR_IN,"--output_dir",OUTPUT_DIR,
   "--data_folder",f"{DATA_DIR}/patient_sequence","--dataset_prepared_path",DATASET_PREP,"--preprocessing_num_workers","16",
   "--do_train","true","--seed","42","--hidden_size","768","--num_hidden_layers","12",
   "--max_position_embeddings",str(MODEL_CTX_LEN),"--sample_packing","--max_tokens_per_batch",MAX_TOK,
   "--include_values","true",
   "--num_train_epochs",str(NUM_EPOCHS),"--gradient_accumulation_steps","1",
   "--learning_rate","0.0002","--warmup_ratio","0.02","--weight_decay","0.01","--lr_scheduler_type","cosine",
   "--save_strategy","steps","--save_steps",str(SAVE_STEPS),"--save_total_limit","2",
   "--evaluation_strategy","no","--load_best_model_at_end","false","--use_early_stopping","false",
   "--logging_steps","20","--report_to","none","--exclude_position_ids","false",
   "--dataloader_num_workers","4" if HAS_GPU else "0", *precision]
if RESUME:
    cks = sorted(Path(OUTPUT_DIR).glob("checkpoint-*"), key=lambda p:int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1)
    if cks: cmd += ["--resume_from_checkpoint", str(cks[-1])]; print("resuming from", cks[-1])

LOGDIR=Path("/home/user/repo/logs"); LOGDIR.mkdir(exist_ok=True)
log_path=LOGDIR/f"train_full-{time.strftime('%Y%m%d-%H%M%S')}.log"
KEEP=re.compile(r'error|exception|traceback|loss|epoch|train_runtime|steps_per_second|saving|checkpoint', re.I)
print(f"\nbatch {MAX_TOK} tok | epochs {NUM_EPOCHS} | save every {SAVE_STEPS}\nlog: {log_path}\n")
n=0; t0=time.time()
with open(log_path,"w") as lf:
    p=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        lf.write(line); n+=1
        if KEEP.search(line): print(line, end="")
        elif n%500==0: print(f"  ... {n} lines, {(time.time()-t0)/60:.1f} min", flush=True)
rc=p.wait(); print(f"\nrc={rc}  ({(time.time()-t0)/60:.1f} min, log -> {log_path})")
if rc!=0: print("".join(open(log_path).readlines()[-30:]))

In [ ]:
import os, sys, glob, shutil, re, time, subprocess, torch
import pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc
from pathlib import Path

# ---- knobs: tuned to finish in ~2h ----
N_FILES, MAX_LEN, NUM_EPOCHS, SAVE_STEPS = 100, 2048, 1, 500
DATA="/home/user/repo/cehrgpt_output_full"; MODEL="/home/user/repo/model_run_full"
SMALL=f"{DATA}/patient_sequence_small"; PREP=f"{DATA}/dataset_prepared_small"; OUT=MODEL+"_real"

# 1) build a filtered subset: first N_FILES files, sequences <= MAX_LEN tokens (drops the giants that stall prep)
shutil.rmtree(SMALL, ignore_errors=True); Path(SMALL).mkdir(parents=True)
kept=tot=0
for f in sorted(glob.glob(f"{DATA}/patient_sequence/*.parquet"))[:N_FILES]:
    t=pq.read_table(f); sub=t.filter(pc.less_equal(t["num_of_concepts"], MAX_LEN))
    pq.write_table(sub, f"{SMALL}/{os.path.basename(f)}"); kept+=sub.num_rows; tot+=t.num_rows


print(f"subset: {kept:,} of {tot:,} sequences (<= {MAX_LEN} tok) from {N_FILES} files")

# 2) fresh prep + train on the subset
shutil.rmtree(PREP, ignore_errors=True); shutil.rmtree(OUT, ignore_errors=True); Path(OUT).mkdir(parents=True)
HAS_GPU=torch.cuda.is_available(); print("GPU:", HAS_GPU, torch.cuda.get_device_name(0) if HAS_GPU else "")
prec=["--bf16","true","--fp16","false"] if (HAS_GPU and torch.cuda.is_bf16_supported()) else ["--bf16","false","--fp16","false"]
env=os.environ.copy()
env.update(OMP_NUM_THREADS="8", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
           TOKENIZERS_PARALLELISM="false", WANDB_DISABLED="true", TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD="1")
if not HAS_GPU: env["CUDA_VISIBLE_DEVICES"]=""
env["LD_LIBRARY_PATH"]=f"{sys.prefix}/lib:"+env.get("LD_LIBRARY_PATH","")
env["PYTHONPATH"]=":".join(["/home/user/repo/cehrbert-main","/home/user/repo/cehrgpt-main/src","/home/user/repo/cehrbert_data-main/src",env.get("PYTHONPATH","")])
cmd=[sys.executable,"-u","-m","cehrgpt.runners.hf_cehrgpt_pretrain_runner",
  "--model_name_or_path",MODEL,"--tokenizer_name_or_path",MODEL,"--output_dir",OUT,
  "--data_folder",SMALL,"--dataset_prepared_path",PREP,"--preprocessing_num_workers","16",
  "--do_train","true","--seed","42","--hidden_size","768","--num_hidden_layers","12",
  "--max_position_embeddings","1024","--sample_packing","--max_tokens_per_batch","4096",
  "--num_train_epochs",str(NUM_EPOCHS),"--gradient_accumulation_steps","1",
  "--learning_rate","0.0002","--warmup_ratio","0.02","--weight_decay","0.01","--lr_scheduler_type","cosine",
  "--save_strategy","steps","--save_steps",str(SAVE_STEPS),"--save_total_limit","2",
  "--evaluation_strategy","no","--load_best_model_at_end","false","--use_early_stopping","false",
  "--logging_steps","20","--report_to","none","--exclude_position_ids","false",
  "--dataloader_num_workers","4" if HAS_GPU else "0", *prec]
LOGDIR=Path("/home/user/repo/logs"); LOGDIR.mkdir(exist_ok=True)
log_path=LOGDIR/f"train_small-{time.strftime('%Y%m%d-%H%M%S')}.log"
KEEP=re.compile(r'error|exception|traceback|loss|epoch|train_runtime|steps_per_second|saving|checkpoint|Map ', re.I)
print("log:", log_path, "\n"); n=0; t0=time.time()
with open(log_path,"w") as lf:
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
    for line in p.stdout:
        lf.write(line); n+=1
        if KEEP.search(line): print(line,end="")
        elif n%500==0: print(f"  ...{n} lines, {(time.time()-t0)/60:.1f} min",flush=True)
rc=p.wait(); print(f"\nrc={rc} ({(time.time()-t0)/60:.1f} min)")
print("✅ TRAINED — model in "+OUT if rc==0 else "❌ see log")

In [ ]:
import os, sys, glob, subprocess
from pathlib import Path
OUT="/home/user/repo/synthetic_small"; Path(OUT).mkdir(parents=True, exist_ok=True)
env=os.environ.copy()
env.update(TOKENIZERS_PARALLELISM="false", WANDB_DISABLED="true", TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD="1")
env["LD_LIBRARY_PATH"]=f"{sys.prefix}/lib:"+env.get("LD_LIBRARY_PATH","")
env["PYTHONPATH"]=":".join(["/home/user/repo/cehrbert-main","/home/user/repo/cehrgpt-main/src","/home/user/repo/cehrbert_data-main/src",env.get("PYTHONPATH","")])
cmd=[sys.executable,"-u","-m","cehrgpt.generation.generate_batch_hf_gpt_sequence",
  "--model_folder","/home/user/repo/model_run_full_real","--tokenizer_folder","/home/user/repo/model_run_full",
  "--output_folder",OUT,"--num_of_patients","100","--batch_size","8","--buffer_size","32","--context_window","1024",
  "--sampling_strategy","TopPStrategy","--top_p","0.95","--temperature","0.9","--repetition_penalty","1.05",
  "--epsilon_cutoff","0.0","--demographic_data_path","/home/user/repo/cehrgpt_output_full/patient_sequence_small"]
r=subprocess.run(cmd,capture_output=True,text=True,env=env)
print("\n".join((r.stdout+r.stderr).splitlines()[-12:]))
g=glob.glob(f"{OUT}/**/*.parquet",recursive=True)
if r.returncode==0 and g:
    import pyarrow.parquet as pq
    print(f"\n✅ generated {sum(pq.read_metadata(x).num_rows for x in g)} synthetic patients -> {OUT}")
else: print("rc=",r.returncode)

In [ ]:
import os, sys, glob, shutil, re, time, subprocess
from pathlib import Path

OMOP_DIR   = "/home/user/repo/omop_synthea"
SYNTH_ROOT = "/home/user/repo/synthetic_small"          # ← your generation output

# locate the generated sequences (robust to nesting depth)
gen = glob.glob(f"{SYNTH_ROOT}/**/generated_sequences", recursive=True)
assert gen, f"no generated_sequences under {SYNTH_ROOT} — run the generation cell first"
GENERATED = gen[0]
RESTORED  = os.path.join(os.path.dirname(GENERATED), "restored_omop")
shutil.rmtree(RESTORED, ignore_errors=True); os.makedirs(RESTORED, exist_ok=True)

CONCEPT = f"{OMOP_DIR}/concept"
n_concept = len(glob.glob(CONCEPT + "/*.parquet"))
if n_concept == 0:
    hits = sorted({os.path.dirname(h) for h in glob.glob("/home/user/repo/**/concept/*.parquet", recursive=True)})
    raise FileNotFoundError(f"concept vocab not found at {CONCEPT}." +
        ("\n  found here instead:\n    " + "\n    ".join(hits) if hits else "  re-run the concept-restore cell first."))
print(f"generated: {GENERATED}  ({len(glob.glob(GENERATED+'/*.parquet'))} patient files)")
print(f"concept  : {CONCEPT}  ({n_concept} files)")

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = f"{sys.prefix}/lib:" + env.get("LD_LIBRARY_PATH","")
env["PYTHONPATH"] = ":".join(["/home/user/repo/cehrbert-main","/home/user/repo/cehrgpt-main/src",
                              "/home/user/repo/cehrbert_data-main/src", env.get("PYTHONPATH","")])
cmd = [sys.executable, "-u", "-m", "cehrgpt.generation.omop_converter_batch",
   "--patient_sequence_path", GENERATED, "--output_folder", RESTORED, "--concept_path", CONCEPT,
   "--cpu_cores", "1", "--buffer_size", "1"]
LOGDIR = Path("/home/user/repo/logs"); LOGDIR.mkdir(exist_ok=True)
log_path = LOGDIR / f"omop_convert-{time.strftime('%Y%m%d-%H%M%S')}.log"
KEEP = re.compile(r'error|exception|traceback|restored|wrote|convert|table|person|visit|condition', re.I)
print(f"logfile: {log_path}\n")
n=0; t0=time.time()
with open(log_path,"w") as lf:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        lf.write(line); n+=1
        if KEEP.search(line): print(line, end="")
        elif n % 200 == 0: print(f"  ... {n} lines, {(time.time()-t0)/60:.1f} min", flush=True)
rc=p.wait()
print(f"\nrc={rc}  ({(time.time()-t0)/60:.1f} min, log -> {log_path})")
if rc==0:
    print("\n--- restored OMOP tables ---")
    for d in sorted(p for p in Path(RESTORED).iterdir() if p.is_dir()):
        ps = list(d.glob("*.parquet"))
        print(f"  {d.name:28s} {len(ps):3d} files  {sum(pf.stat().st_size for pf in ps)/1024:8.1f} KB")
else:
    print("".join(open(log_path).readlines()[-30:]))

In [ ]:
import os, subprocess
p = "/home/user/repo/synthetic_omop_small.tar.gz"
print(p, f"{os.path.getsize(p)/1e6:.2f} MB" if os.path.exists(p) else "MISSING")
print(subprocess.run(["bash","-lc", f"tar tzf {p} | head -20"], capture_output=True, text=True).stdout)

In [ ]:
import glob, pyarrow.parquet as pq, pandas as pd
ROOT = glob.glob("/home/user/repo/synthetic_small/**/restored_omop", recursive=True)[0]
def load(t):
    fs=glob.glob(f"{ROOT}/{t}/*.parquet")
    return pd.concat([pq.read_table(f).to_pandas() for f in fs], ignore_index=True) if fs else pd.DataFrame()
person, visit, cond = load("person"), load("visit_occurrence"), load("condition_occurrence")
pid = person.iloc[0]["person_id"]
print("PATIENTS:", len(person), "| VISITS:", len(visit), "| CONDITIONS:", len(cond))
print("\n=== sample synthetic patient ===\n", person[person.person_id==pid].T)
print("\n--- their visits ---\n", visit[visit.person_id==pid].head(8).to_string())
print("\n--- their conditions ---\n", cond[cond.person_id==pid].head(12).to_string())

In [ ]:
from foundry.transforms import Dataset
TAR  = "/home/user/repo/synthetic_omop_small.tar.gz"
NAME = "copd_synthetic_omop"        # must exist AND kernel must have started after you created it
Dataset.get(NAME).upload_file(TAR, "synthetic_omop_small.tar.gz")
print("✓ uploaded to", NAME)

In [ ]:
import glob, os
import pyarrow.parquet as pq
from collections import defaultdict

ROOT = glob.glob("/home/user/repo/synthetic_small/**/restored_omop", recursive=True)[0]
print("restored_omop:", ROOT, "\n")
for tbl_dir in sorted(glob.glob(f"{ROOT}/*")):
    if not os.path.isdir(tbl_dir):
        continue
    name = os.path.basename(tbl_dir)
    files = sorted(glob.glob(f"{tbl_dir}/*.parquet"))
    if not files:
        print(f"=== {name}: EMPTY (0 files) ===\n"); continue
    rows = sum(pq.ParquetFile(f).metadata.num_rows for f in files)
    coltypes = defaultdict(set)
    for f in files:
        for fld in pq.read_schema(f):
            coltypes[fld.name].add(str(fld.type))
    print(f"=== {name}: {len(files)} files, {rows:,} rows ===")
    for col, ts in coltypes.items():
        print(f"    {col:34s} {' | '.join(sorted(ts))}{'   ⚠ DRIFT' if len(ts) > 1 else ''}")
    print()

In [ ]:
import glob, os, subprocess
import pyarrow as pa, pyarrow.parquet as pq
from collections import defaultdict
from foundry.transforms import Dataset

ROOT = glob.glob("/home/user/repo/synthetic_small/**/restored_omop", recursive=True)[0]
ALIASES = {
    "person":"copd_synth_person", 
    "visit_occurrence":"copd_synth_visit_occurrence",
    "condition_occurrence":"copd_synth_condition_occurrence", 
    "drug_exposure":"copd_synth_drug_exposure",
    "procedure_occurrence":"copd_synth_procedure_occurrence", 
    "death":"copd_synth_death",
}
NULL_FIX = {"refills":pa.int64(), "quantity":pa.float64(), "days_supply":pa.int64()}
ROWS_PER_PART = 500_000
STAGE = "/home/user/repo/_omop_upload"

def unified_schema(files):
    # metadata only; sample the ends at huge scale so we never read millions of footers
    sample = files if len(files) <= 120 else files[:60] + files[-60:]
    cols = defaultdict(list); order = []
    for f in sample:
        for fld in pq.read_schema(f):
            if fld.name not in cols: order.append(fld.name)
            cols[fld.name].append(fld.type)
    out = []
    for n in order:
        nn = [t for t in cols[n] if not pa.types.is_null(t)]
        if not nn:                                          t = NULL_FIX.get(n, pa.float64())
        elif len({str(x) for x in nn}) == 1:                t = nn[0]
        elif any(pa.types.is_floating(x) for x in nn):      t = pa.float64()
        elif any(pa.types.is_string(x) for x in nn):        t = pa.string()
        elif all(pa.types.is_integer(x) for x in nn):       t = pa.int64()
        else:                                               t = nn[0]
        out.append(pa.field(n, t))
    return pa.schema(out)

for tbl, alias in ALIASES.items():
    files = sorted(glob.glob(f"{ROOT}/{tbl}/*.parquet"))
    if not files: print("·  skip", tbl, "(no files)"); continue
    sch = unified_schema(files)
    outdir = f"{STAGE}/{tbl}"; subprocess.run(["bash","-lc", f"rm -rf {outdir} && mkdir -p {outdir}"], check=True)
    writer=None; part=0; in_part=0; total=0; bad=0
    for f in files:                                          # one input file at a time
        try:
            for b in pq.ParquetFile(f).iter_batches(batch_size=20_000):   # one batch at a time
                t = pa.Table.from_batches([b]).select(sch.names).cast(sch, safe=False)
                if writer is None: writer = pq.ParquetWriter(f"{outdir}/part-{part:05d}.parquet", sch)
                writer.write_table(t); in_part += t.num_rows; total += t.num_rows
                if in_part >= ROWS_PER_PART: writer.close(); writer=None; part+=1; in_part=0
        except Exception as e:
            bad += 1
            if bad <= 3: print(f"   skip {os.path.basename(f)}: {repr(e)[:70]}")
    if writer: writer.close()
    parts = sorted(glob.glob(f"{outdir}/*.parquet"))
    ds = Dataset.get(alias)
    for p in parts: ds.upload_file(p, os.path.basename(p))   # file-level upload, no RAM load
    print(f"✓ {tbl:22s} -> {alias}  ({total:,} rows, {len(parts)} part(s)" + (f", {bad} skipped)" if bad else ")"))

## Step 3 — Generate synthetic sequences (full)
Load the trained model + tokenizer and sample synthetic patients. Scale `--num_of_patients` up for the real cohort.


In [ ]:
import os, glob, subprocess
import pyarrow.parquet as pq
from foundry.transforms import Dataset

OUT="/home/user/repo/synthetic_full_1M"
subprocess.run(["bash","-lc",f"rm -rf {OUT} && mkdir -p {OUT}"],check=True)
tars=[p for p in Dataset.get("copd_synthetic_v2").files().download().values() if str(p).endswith(".tar")]
assert tars, "copd_synthetic_v2 has no tar — nothing was banked!"
subprocess.run(["bash","-lc",f"tar xf {max(tars,key=os.path.getsize)} -C {OUT}"],check=True)

files=glob.glob(f"{OUT}/**/generated_sequences/*.parquet",recursive=True)
have=sum(pq.read_metadata(f).num_rows for f in files)
remaining=max(0, 1_000_000-have)
lens=[len(s or []) for s in pq.read_table(files[0],columns=["concept_ids"])["concept_ids"].to_pylist()]
print(f"restored {len(files)} shards, {have:,} patients | remaining: {remaining:,}")
print(f"seq lengths: max {max(lens)}, median {sorted(lens)[len(lens)//2]}   (expect max ≈ context_window 1024)")

In [ ]:
import os, sys, subprocess, time
SYNTH_OUT="/home/user/repo/synthetic_full_1M"; DATA_DIR="/home/user/repo/cehrgpt_output_full"
env=os.environ.copy()
env["OMP_NUM_THREADS"]="8"; env["TOKENIZERS_PARALLELISM"]="false"; env["WANDB_DISABLED"]="true"
env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"]="1"; env["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
env["LD_LIBRARY_PATH"]=f"{sys.prefix}/lib:"+env.get("LD_LIBRARY_PATH","")
env["PYTHONPATH"]=":".join(["/home/user/repo/cehrbert-main","/home/user/repo/cehrgpt-main/src",
                            "/home/user/repo/cehrbert_data-main/src",env.get("PYTHONPATH","")])
cmd=[sys.executable,"-u","-m","cehrgpt.generation.generate_batch_hf_gpt_sequence",
   "--model_folder","/home/user/repo/model_run_full_real","--tokenizer_folder","/home/user/repo/model_run_full",
   "--output_folder",SYNTH_OUT,"--num_of_patients",str(remaining),
   "--batch_size","256","--buffer_size","1024","--context_window","1024",
   "--sampling_strategy","TopPStrategy","--top_p","0.95","--temperature","0.9",
   "--repetition_penalty","1.05","--epsilon_cutoff","0.00","--demographic_data_path",f"{DATA_DIR}/patient_sequence"]
log=f"/home/user/repo/logs/gen1M-1024-{time.strftime('%Y%m%d-%H%M%S')}.log"
subprocess.run(["bash","-lc","pkill -9 -f generate_batch_hf_gpt_sequence 2>/dev/null; true"])
p=subprocess.Popen(cmd,stdout=open(log,"w"),stderr=subprocess.STDOUT,start_new_session=True,env=env)
print(f"resumed ctx-1024 for {remaining:,} more, PID {p.pid}\nlog: {log}")

In [ ]:
import subprocess, os, glob, json
from foundry.transforms import Dataset

tars=[p for p in Dataset.get("copd_cehrgpt_v2_resume").files().download().values() if str(p).endswith(".tar")]
subprocess.run(["bash","-lc",
  f"rm -rf /home/user/repo/model_run_full /home/user/repo/model_run_full_real; "
  f"tar xf {max(tars,key=os.path.getsize)} -C /home/user/repo"], check=True)

ps="/home/user/repo/cehrgpt_output_full/patient_sequence"
subprocess.run(["bash","-lc",f"rm -rf {ps} && mkdir -p {ps}"],check=True)
dtar=max(Dataset.get("copd_patient_sequence_le2048").files().download().values(), key=os.path.getsize)
subprocess.run(["bash","-lc",f"tar xf {dtar} -C {ps}"],check=True)

cfg=json.load(open("/home/user/repo/model_run_full_real/config.json"))
print("vocab_size:", cfg.get("vocab_size"),
      "| model.safetensors:", os.path.exists("/home/user/repo/model_run_full_real/model.safetensors"),
      "| tokenizer:", os.path.exists("/home/user/repo/model_run_full/cehrgpt_tokenizer.json"),
      "| seed files:", len(glob.glob(f"{ps}/*.parquet")))

In [ ]:
f = "/home/user/repo/cehrgpt-main/src/cehrgpt/models/tokenization_hf_cehrgpt.py"
src = open(f).read()
old = "encoded = self._tokenizer.encode(concept_ids, is_pretokenized=True)"
new = "encoded = self._tokenizer.encode([str(c) for c in concept_ids], is_pretokenized=True)"
if new in src:
    print("already patched")
elif old in src:
    open(f,"w").write(src.replace(old, new))
    print("✓ patched encode() to coerce concept_ids -> List[str]")
else:
    print("target line not found — inspect encode() in tokenization_hf_cehrgpt.py; upstream may have changed")

In [ ]:
import re
f="/home/user/repo/cehrgpt-main/src/cehrgpt/gpt_utils.py"
src=open(f).read()
patched=0
for fn in ["_orig_construct_age_sequence","_orig_construct_time_sequence"]:
    pat=re.compile(rf"({re.escape(fn)}\()concept_ids([,)])")
    src,n=pat.subn(r"\1[('0' if c is None else str(c)) for c in concept_ids]\2", src)
    patched+=n
open(f,"w").write(src)
print(f"sanitized concept_ids input in {patched} wrapper call(s)")

In [ ]:
import os, glob, subprocess
import pyarrow.parquet as pq
from foundry.transforms import Dataset

OUT="/home/user/repo/synthetic_full_1M"
subprocess.run(["bash","-lc",f"rm -rf {OUT} && mkdir -p {OUT}"],check=True)
tars=[p for p in Dataset.get("copd_synthetic_v2").files().download().values() if str(p).endswith(".tar")]
assert tars, "copd_synthetic_v2 has no tar — nothing was banked!"
subprocess.run(["bash","-lc",f"tar xf {max(tars,key=os.path.getsize)} -C {OUT}"],check=True)

files=glob.glob(f"{OUT}/**/generated_sequences/*.parquet",recursive=True)
have=sum(pq.read_metadata(f).num_rows for f in files)
remaining=max(0, 1_000_000-have)
lens=[len(s or []) for s in pq.read_table(files[0],columns=["concept_ids"])["concept_ids"].to_pylist()]
print(f"restored {len(files)} shards, {have:,} patients | remaining: {remaining:,}")
print(f"seq lengths: max {max(lens)}, median {sorted(lens)[len(lens)//2]}   (expect max ≈ context_window 1024)")

In [ ]:
import os, sys, subprocess, time
SYNTH_OUT="/home/user/repo/synthetic_full_1M"; DATA_DIR="/home/user/repo/cehrgpt_output_full"
env=os.environ.copy()
env["OMP_NUM_THREADS"]="8"; env["TOKENIZERS_PARALLELISM"]="false"; env["WANDB_DISABLED"]="true"
env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"]="1"; env["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
env["LD_LIBRARY_PATH"]=f"{sys.prefix}/lib:"+env.get("LD_LIBRARY_PATH","")
env["PYTHONPATH"]=":".join(["/home/user/repo/cehrbert-main","/home/user/repo/cehrgpt-main/src",
                            "/home/user/repo/cehrbert_data-main/src",env.get("PYTHONPATH","")])
cmd=[sys.executable,"-u","-m","cehrgpt.generation.generate_batch_hf_gpt_sequence",
   "--model_folder","/home/user/repo/model_run_full_real","--tokenizer_folder","/home/user/repo/model_run_full",
   "--output_folder",SYNTH_OUT,"--num_of_patients",str(remaining),
   "--batch_size","256","--buffer_size","1024","--context_window","1024",
   "--sampling_strategy","TopPStrategy","--top_p","0.95","--temperature","0.9",
   "--repetition_penalty","1.05","--epsilon_cutoff","0.00","--demographic_data_path",f"{DATA_DIR}/patient_sequence"]
log=f"/home/user/repo/logs/gen1M-1024-{time.strftime('%Y%m%d-%H%M%S')}.log"
subprocess.run(["bash","-lc","pkill -9 -f generate_batch_hf_gpt_sequence 2>/dev/null; true"])
p=subprocess.Popen(cmd,stdout=open(log,"w"),stderr=subprocess.STDOUT,start_new_session=True,env=env)
print(f"resumed ctx-1024 for {remaining:,} more, PID {p.pid}\nlog: {log}")

In [ ]:
import subprocess, sys, textwrap, time
from pathlib import Path
W="/home/user/repo/gen_bank_watcher.py"
Path(W).write_text(textwrap.dedent('''
    import time, glob, os, subprocess
    from foundry.transforms import Dataset
    OUT="/home/user/repo/synthetic_full_1M"; DS="copd_synthetic_v2"; INTERVAL=600
    print(time.strftime("%H:%M:%S"),"gen-bank watcher up ->",DS,flush=True)
    while True:
        try:
            n=len(glob.glob(OUT+"/**/generated_sequences/*.parquet",recursive=True))
            if n>0:
                tar="/home/user/repo/synthetic_v2.tar"
                subprocess.run(["bash","-lc",f"cd {OUT} && tar cf {tar} . ; ec=$?; [ $ec -le 1 ]"],check=True)
                Dataset.get(DS).upload_file(tar,"synthetic_v2.tar")
                print(time.strftime("%H:%M:%S"),"BANKED",n,"shards",f"({os.path.getsize(tar)/1e9:.2f} GB)",flush=True)
        except Exception as e:
            print(time.strftime("%H:%M:%S"),"bank err:",repr(e)[:140],flush=True)
        time.sleep(INTERVAL)
    '''))
subprocess.run(["bash","-lc","pkill -f gen_bank_watcher.py 2>/dev/null; true"]); time.sleep(1)
p=subprocess.Popen([sys.executable,"-u",W],stdout=open("/home/user/repo/logs/gen_bank_watcher.log","a"),
                   stderr=subprocess.STDOUT,start_new_session=True)
print("watcher PID",p.pid)

In [ ]:
import subprocess
print(subprocess.run(["bash","-lc","tail -5 /home/user/repo/logs/gen_bank_watcher.log"],capture_output=True,text=True).stdout)

In [ ]:
import glob, time, subprocess, pyarrow.parquet as pq
OUT="/home/user/repo/synthetic_full_1M"
def count():
    fs=glob.glob(OUT+"/**/generated_sequences/*.parquet",recursive=True)
    return sum(pq.read_metadata(f).num_rows for f in fs) if fs else 0
print("proc alive:", subprocess.run(["bash","-lc","pgrep -fc generate_batch_hf_gpt_sequence"],capture_output=True,text=True).stdout.strip())
print("GPU:", subprocess.run(["bash","-lc","nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader"],capture_output=True,text=True).stdout.strip())
a=count(); time.sleep(60); b=count(); rate=(b-a)/60
print(f"generated: {b:,} | rate: {rate:.1f}/sec | ETA 1M: {(1_000_000-b)/rate/3600:.1f} h" if rate>0
      else f"generated: {b} — no new rows in 60s (still loading seeds; wait, rerun)")

In [ ]:
import glob, random, collections, statistics
import pyarrow.parquet as pq

OUT="/home/user/repo/synthetic_full_1M"
files=sorted(glob.glob(f"{OUT}/**/generated_sequences/*.parquet",recursive=True))
total=sum(pq.read_metadata(f).num_rows for f in files)
print(f"=== {len(files)} shards, {total:,} patients total ===\n")

# sample ~50 shards for the detailed stats (memory-safe)
random.seed(42)
sample=random.sample(files,min(50,len(files)))
lens=[]; death_pat=0; meas_pat=0; val_pat=0; years=collections.Counter()
vs_bal=collections.Counter(); pids=set(); dup=0; n=0
DEATH={"4306655","4304866","4267568","443768"}
for f in sample:
    t=pq.read_table(f,columns=["concept_ids","person_id","concept_value_masks","number_as_values"])
    for seq,pid,m,nv in zip(t["concept_ids"].to_pylist(),t["person_id"].to_pylist(),
                            t["concept_value_masks"].to_pylist(),t["number_as_values"].to_pylist()):
        n+=1; s=[str(x) for x in (seq or [])]
        lens.append(len(s))
        if pid in pids: dup+=1
        pids.add(pid)
        if any(x in DEATH or "death" in x.lower() for x in s): death_pat+=1
        # measurement concepts: OMOP LOINC-range ids commonly 3xxxxxx
        if any(x.isdigit() and x.startswith("3") and len(x)==7 for x in s): meas_pat+=1
        if m and sum(int(v or 0) for v in m)>0: val_pat+=1
        for x in s:
            if x.startswith("year:"): years[x]+=1
        vs_bal["VS"]+=s.count("[VS]"); vs_bal["VE"]+=s.count("[VE]")

lens.sort()
def pct(p): return lens[int(len(lens)*p)]
print(f"--- sequence lengths (n={n:,} sampled) ---")
print(f"min {lens[0]} | p25 {pct(.25)} | median {pct(.5)} | mean {statistics.mean(lens):.0f} | p90 {pct(.9)} | max {lens[-1]}")
print(f"\n--- death ---\npatients with death token: {death_pat}/{n}")
print(f"\n--- measurements ---")
print(f"patients with measurement concepts (LOINC 3xxxxxx): {meas_pat}/{n} ({100*meas_pat/n:.0f}%)")
print(f"patients with numeric lab VALUES: {val_pat}/{n}")
print(f"\n--- structure ---")
print(f"[VS]/[VE] balance: {vs_bal['VS']:,} / {vs_bal['VE']:,} (should be ~equal)")
print(f"duplicate person_ids in sample: {dup}")
print(f"\n--- start years (top 10) ---")
for y,c in years.most_common(10): print(f"  {y}: {c}")

In [ ]:
import glob, collections, statistics, subprocess, pyarrow.parquet as pq
subprocess.run(["bash","-lc","pkill -f gen_bank_watcher.py 2>/dev/null; pkill -9 -f generate_batch_hf_gpt_sequence 2>/dev/null; true"])  # gen's done

SEED="/home/user/repo/cehrgpt_output_full/patient_sequence"
CAP=2000
dist=collections.defaultdict(list)                       # concept -> real values
unit_cnt=collections.defaultdict(collections.Counter)    # concept -> unit votes
files=sorted(glob.glob(f"{SEED}/*.parquet"))
for k,f in enumerate(files):
    t=pq.read_table(f, columns=["concept_ids","concept_value_masks","number_as_values","units"])
    ci=t["concept_ids"].to_pylist(); mk=t["concept_value_masks"].to_pylist()
    nv=t["number_as_values"].to_pylist(); un=t["units"].to_pylist()
    for a,m,v,u in zip(ci,mk,nv,un):
        if not a or not m or not v: continue
        for i in range(min(len(a),len(m),len(v))):
            if m[i] and v[i] is not None:
                c=str(a[i])
                if len(dist[c])<CAP: dist[c].append(float(v[i]))
                if u and i<len(u) and u[i]: unit_cnt[c][str(u[i])]+=1
    if k%400==0: print(f"  scanned {k}/{len(files)} files, {len(dist)} concepts so far")
unit_mode={c:cnt.most_common(1)[0][0] for c,cnt in unit_cnt.items()}
print(f"\nbuilt distributions for {len(dist)} lab concepts")
for c in list(dist)[:4]:
    v=dist[c]; print(f"  {c}: n={len(v)} mean={statistics.mean(v):.2f} range[{min(v):.1f},{max(v):.1f}] unit={unit_mode.get(c,'')}")

In [ ]:
import glob, random, pyarrow as pa, pyarrow.parquet as pq
random.seed(1)
GEN="/home/user/repo/synthetic_full_1M"
files=glob.glob(f"{GEN}/**/generated_sequences/*.parquet",recursive=True)
dist={c:[x for x in vs if x==x and abs(x)!=float('inf')] for c,vs in dist.items()}
concepts={c for c in dist if dist[c]}
# generated value columns are list<null> (empty) — give them real types
MT=pa.list_(pa.int32()); VT=pa.list_(pa.float32()); IT=pa.list_(pa.int32()); UT=pa.list_(pa.string())
inj=tot=0
for j,f in enumerate(files):
    t=pq.read_table(f); ci_col=t["concept_ids"].to_pylist()
    masks=[]; vals=[]; nts=[]; units=[]
    for ci in ci_col:
        L=len(ci); m=[0]*L; v=[0.0]*L; nt=[0]*L; u=[None]*L
        for i in range(L):
            c=str(ci[i])
            if c in concepts:
                m[i]=1; v[i]=float(random.choice(dist[c])); nt[i]=1; u[i]=unit_mode.get(c); inj+=1
        masks.append(m); vals.append(v); nts.append(nt); units.append(u); tot+=1
    cols={n:t[n] for n in t.schema.names}
    cols["concept_value_masks"]=pa.array(masks,type=MT)
    cols["number_as_values"]=pa.array(vals,type=VT)
    cols["is_numeric_types"]=pa.array(nts,type=IT)
    cols["units"]=pa.array(units,type=UT)
    pq.write_table(pa.table(cols), f)
    if j%200==0: print(f"  {j}/{len(files)} shards")
print(f"\n✓ injected values into {inj:,} measurement positions across {tot:,} patients")

In [ ]:
import glob, random, pyarrow.parquet as pq
files=glob.glob("/home/user/repo/synthetic_full_1M/**/generated_sequences/*.parquet",recursive=True)
random.seed(2); samp=random.sample(files,min(30,len(files)))
tot=withval=nval=0
for f in samp:
    t=pq.read_table(f,columns=["concept_ids","concept_value_masks","number_as_values"])
    for m,nv in zip(t["concept_value_masks"].to_pylist(),t["number_as_values"].to_pylist()):
        tot+=1
        if m and sum(int(x or 0) for x in m)>0: withval+=1
        if nv: nval+=sum(1 for x in nv if x not in (None,0,0.0))
print(f"{tot:,} sampled | with >=1 lab value: {withval} ({100*withval/tot:.0f}%) | total values: {nval:,}")
r=pq.read_table(samp[0]); row=r.slice(0,1).to_pylist()[0]
labs=[(row['concept_ids'][i],row['number_as_values'][i],row['units'][i])
      for i in range(len(row['concept_ids'])) if row['concept_value_masks'][i]]
print("sample patient (concept, value, unit):", labs[:12])

## Step 4 — Convert synthetic sequences to OMOP (full)
Routes generated concepts back into `person` / `visit_occurrence` / `condition_occurrence` / `drug_exposure` / `procedure_occurrence`.


In [ ]:
from foundry.transforms import Dataset
import os, shutil, glob
from pathlib import Path

OMOP_DIR = "/home/user/repo/omop_synthea"
out = Path(OMOP_DIR) / "concept"
out.mkdir(parents=True, exist_ok=True)
for f in out.glob("*.parquet"): f.unlink()          # clear any stale copy

downloaded = Dataset.get("copd_concept_parquet").files().download()
srcs = [Path(p) for p in downloaded.values()]
parquet = [p for p in srcs if p.suffix == ".parquet"] or [p for p in srcs if p.is_file()]
if not parquet:
    print("download returned no files:", [p.name for p in srcs][:10])
n = 0
for src in parquet:
    shutil.copy(src, out / src.name); n += 1

print(f"restored {n} concept file(s), "
      f"{len(glob.glob(f'{out}/*.parquet'))} parquet now present -> {out}")

In [ ]:
import glob, collections, random, pyarrow.parquet as pq
GEN="/home/user/repo/synthetic_full_1M"
files=glob.glob(f"{GEN}/**/generated_sequences/*.parquet",recursive=True)
random.seed(0); samp=random.sample(files, min(40,len(files)))
tot=badage=short=0; p1=collections.Counter()
for f in samp:
    for seq in pq.read_table(f,columns=["concept_ids"])["concept_ids"].to_pylist():
        s=[str(x) for x in (seq or [])]; tot+=1
        if len(s)>1: p1[s[1]]+=1
        if len(s)<=1 or s[1]=="[OOV]" or not s[1].startswith("age"): badage+=1
        if len(s)<8: short+=1
print(f"{tot:,} sampled")
print(f"  bad/OOV age (pos 1)              : {badage} ({100*badage/tot:.1f}%)")
print(f"  very short <8 tokens (no events) : {short} ({100*short/tot:.1f}%)")
print(f"  position-1 token dist: {p1.most_common(8)}")

In [ ]:
import os, sys, glob, shutil, re, time, subprocess
from pathlib import Path
import pyarrow.parquet as pq

OMOP_DIR="/home/user/repo/omop_synthea"
SYNTH_ROOT="/home/user/repo/synthetic_full_1M"          # full 1M-patient generation output

gen=glob.glob(f"{SYNTH_ROOT}/*/generated_sequences")
assert gen, f"no generated_sequences under {SYNTH_ROOT}"
GENERATED=gen[0]
RESTORED=os.path.join(os.path.dirname(GENERATED),"restored_omop")
shutil.rmtree(RESTORED, ignore_errors=True); os.makedirs(RESTORED, exist_ok=True)

CONCEPT=f"{OMOP_DIR}/concept"
n_concept=len(glob.glob(CONCEPT+"/*.parquet"))
assert n_concept>0, f"concept vocab missing at {CONCEPT} — run the concept restore cell first"
in_seqs=sum(pq.read_metadata(f).num_rows for f in glob.glob(GENERATED+"/*.parquet"))
print(f"generated: {GENERATED} ({len(glob.glob(GENERATED+'/*.parquet'))} files, {in_seqs:,} sequences)")
print(f"concept  : {CONCEPT} ({n_concept} files)")

env=os.environ.copy()
env["LD_LIBRARY_PATH"]=f"{sys.prefix}/lib:"+env.get("LD_LIBRARY_PATH","")
env["PYTHONPATH"]=":".join(["/home/user/repo/cehrbert-main","/home/user/repo/cehrgpt-main/src",
                            "/home/user/repo/cehrbert_data-main/src",env.get("PYTHONPATH","")])
cmd=[sys.executable,"-u","-m","cehrgpt.generation.omop_converter_batch",
   "--patient_sequence_path",GENERATED,"--output_folder",RESTORED,"--concept_path",CONCEPT,
   "--cpu_cores","8","--buffer_size","128"]

LOGDIR=Path("/home/user/repo/logs"); LOGDIR.mkdir(exist_ok=True)
log_path=LOGDIR/f"omop_convert-{time.strftime('%Y%m%d-%H%M%S')}.log"
KEEP=re.compile(r'error|exception|traceback|restored|wrote|convert|person|visit|condition|measurement',re.I)
print(f"log: {log_path}\n"); n=0; t0=time.time()
with open(log_path,"w") as lf:
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
    for line in p.stdout:
        lf.write(line); n+=1
        if KEEP.search(line): print(line,end="")
        elif n%500==0: print(f"  ... {n} lines, {(time.time()-t0)/60:.1f} min",flush=True)
rc=p.wait(); print(f"\nrc={rc} ({(time.time()-t0)/60:.1f} min)")
if rc==0:
    print("\n--- restored OMOP tables ---")
    for d in sorted(Path(RESTORED).iterdir()):
        if d.is_dir():
            ps=list(d.glob("*.parquet"))
            rows=sum(pq.read_metadata(str(x)).num_rows for x in ps) if ps else 0
            print(f"  {d.name:26s} {len(ps):3d} files  {rows:>11,} rows")
    pdir=Path(RESTORED)/"person"
    if pdir.exists():
        persons=sum(pq.read_metadata(str(x)).num_rows for x in pdir.glob("*.parquet"))
        print(f"\nperson: {persons:,} of {in_seqs:,} sequences kept ({100*persons/in_seqs:.0f}%)")
else:
    print("".join(open(log_path).readlines()[-30:]))

In [ ]:
import glob, numpy as np, pandas as pd
RESTORED=glob.glob("/home/user/repo/synthetic_full_1M/*/restored_omop")[0]
print("post-processing:", RESTORED)

person=pd.concat([pd.read_parquet(f) for f in glob.glob(f"{RESTORED}/person/*.parquet")], ignore_index=True)
BYC="year_of_birth" if "year_of_birth" in person.columns else "birth_year"
yob=person.groupby("person_id")[BYC].first()

firsts=[]                                   # first visit per person, streamed (2 cols only)
for f in glob.glob(f"{RESTORED}/visit_occurrence/*.parquet"):
    v=pd.read_parquet(f, columns=["person_id","visit_start_date"])
    firsts.append(v.groupby("person_id")["visit_start_date"].min())
first=pd.to_datetime(pd.concat(firsts).groupby(level=0).min())

fy=first.dt.year
ystart=pd.to_datetime(dict(year=fy,month=1,day=1)); yend=pd.to_datetime(dict(year=fy,month=12,day=31))
byr=yob.reindex(first.index); byr=byr.where(byr.notna(),fy).astype(int)
bstart=pd.to_datetime(dict(year=byr,month=1,day=1))
lo=np.maximum(ystart.values,bstart.values)
span=((yend.values-lo)/np.timedelta64(1,'D')).clip(min=0).astype(int)
rnd=np.floor(np.random.random(len(first))*(span+1)).astype(int)
newf=pd.to_datetime(lo)+pd.to_timedelta(rnd,unit="D")
shift=pd.Series(((newf.values-first.values)/np.timedelta64(1,'D')).astype("int64"),
                index=first.index, name="shift_days")
print(f"computed time-shift for {len(shift):,} persons (median |shift| {int(np.median(np.abs(shift)))} d)")

In [ ]:
import os, glob, numpy as np, pandas as pd
OUT=os.path.join(os.path.dirname(RESTORED),"restored_omop_pp"); os.makedirs(OUT, exist_ok=True)

for tbl in ["visit_occurrence","condition_occurrence","drug_exposure","measurement","procedure_occurrence","death"]:
    files=glob.glob(f"{RESTORED}/{tbl}/*.parquet")
    if not files: continue
    dst=f"{OUT}/{tbl}"; os.makedirs(dst, exist_ok=True); dcols=None
    for f in files:
        df=pd.read_parquet(f)
        if dcols is None: dcols=[c for c in df.columns if "date" in c.lower()]
        td=pd.to_timedelta(df["person_id"].map(shift).fillna(0).astype("int64"), unit="D")
        for c in dcols:
            s=pd.to_datetime(df[c], errors="coerce")+td
            df[c]=s.dt.date if c.lower().endswith("_date") else s
        df.to_parquet(f"{dst}/{os.path.basename(f)}", index=False)
    print(f"  shifted {tbl:24s} {len(files)} files  cols={dcols}")

pdst=f"{OUT}/person"; os.makedirs(pdst, exist_ok=True)
for f in glob.glob(f"{RESTORED}/person/*.parquet"):
    df=pd.read_parquet(f); n=len(df); mob=np.random.randint(1,13,n)
    dob=(np.floor(np.random.random(n)*pd.to_datetime(dict(year=df[BYC],month=mob,day=1)).dt.daysinmonth).astype(int)+1)
    if "month_of_birth" in df.columns: df["month_of_birth"]=mob
    if "day_of_birth" in df.columns: df["day_of_birth"]=dob
    df["birth_datetime"]=pd.to_datetime(dict(year=df[BYC],month=mob,day=dob))
    df.to_parquet(f"{pdst}/{os.path.basename(f)}", index=False)
print("  randomized person birthdates\n✓ ->", OUT)

In [ ]:
import subprocess, glob, os
import pyarrow.parquet as pq
from foundry.transforms import Dataset

PP=glob.glob("/home/user/repo/synthetic_full_1M/*/restored_omop_pp")[0]
NAME="copd_omop_valued"        # create this dataset in the UI + restart kernel if it 404s

# what's in it
print("banking:", PP)
for tbl in sorted(os.listdir(PP)):
    if os.path.isdir(f"{PP}/{tbl}"):
        print(f"  {tbl:24s} {len(glob.glob(f'{PP}/{tbl}/*.parquet'))} files")

# confirm no-values vs valued (sample one measurement file)
mf=glob.glob(f"{PP}/measurement/*.parquet")[0]
if "value_as_number" in pq.read_schema(mf).names:
    col=pq.read_table(mf, columns=["value_as_number"])["value_as_number"]
    print(f"\nmeasurement value_as_number: {len(col)-col.null_count}/{len(col)} non-null "
          f"→ {'VALUES present (rename to copd_omop_valued!)' if (len(col)-col.null_count)>0 else 'no values (correct for this dataset)'}")

try: Dataset.get(NAME).files()
except Exception as e: raise SystemExit(f"create '{NAME}' in the UI + restart kernel first ({repr(e)[:60]})")

TAR="/home/user/repo/omop_novalues_pp.tar"
subprocess.run(["bash","-lc",f"cd {PP} && tar cf {TAR} . ; [ $? -le 1 ] && ls -lh {TAR}"],check=True)
Dataset.get(NAME).upload_file(TAR,"omop_novalues_pp.tar")
print(f"\n✓ banked post-processed OMOP -> {NAME}")

In [ ]:
import subprocess, glob
from foundry.transforms import Dataset
RESTORED=glob.glob("/home/user/repo/synthetic_full_1M/*/restored_omop")[0]
NAME="copd_omop_novalues"
try: Dataset.get(NAME).files()
except Exception as e: raise SystemExit(f"create '{NAME}' in the UI + restart kernel first ({repr(e)[:50]})")
TAR="/home/user/repo/omop_novalues.tar"
subprocess.run(["bash","-lc",f"cd {RESTORED} && tar cf {TAR} . && ls -lh {TAR}"],check=True)
Dataset.get(NAME).upload_file(TAR,"omop_novalues.tar")
print(f"✓ banked no-values OMOP -> {NAME}")